In [ ]:
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn # To ignore some warnings from sklearn logistic regression
from sklearn.exceptions import ConvergenceWarning # To avoid logistic regression failing in case of perfect separation

import datetime
import numpy as np
from scipy.optimize import minimize # minimizing function for AIOLI
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
import pandas as pd
from sklearn.linear_model import LogisticRegression # for classic logistic regression
from typing import Literal
from sklearn.neighbors import KDTree # For Knn for SMOTE

options={"disp": False} # To not get verbose from minimize function
np.set_printoptions(suppress=True, precision=4) #removes e notation
pd.options.display.float_format = '{:.10f}'.format # To remove e notation
from matplotlib.lines import Line2D

# Global variables

In [ ]:
default_n = 100
default_runs = 10
default_test_size = 100

# Algorithms

## Helper functions for algorithms

In [ ]:
def sig(z):
    """ Sigmoid function that avoids overflow """
    if z < 0:
        return np.exp(z) / (1 + np.exp(z))
    else:
        return 1 / (1 + np.exp(-z))

In [ ]:
def calculate_OTB(array, method: Literal["suffix_averaging", "uniform", "last_iterate"] | None = None, p = 0.5):
    """ Weighted average function
    ----
    Parameters:
        p: Proportion of last observations to average over. Ex p = 0.3 means average over last 30% of observations.
    Methods:
        Suffix averaging: Average over last p percentage of observations.
        Uniform: Average over all iterates equally
        Last iterate: Consider only the last iterate.
        """
    n = len(array)
    # Check p is a proportion
    if p is None:
        p = 0.5
    if not (0 <= p <= 1):
        raise ValueError("p must be between 0 and 1")

    match method:
        case "suffix_averaging":
            i = round((1-p)*n)
            result = np.mean(array[i:], axis=0)
        case "last_iterate":
            last_iterate = n - 1
            result = array[last_iterate]
        case _: # Uniform is default
            result = np.mean(array, axis=0)

    return result

In [ ]:
def log_loss(betas,data_x,data_y):
    """ Works for a vector of betas and a whole dataset """
    return np.mean(np.logaddexp(0,-data_y*np.inner(data_x,betas)))

In [ ]:
def generate_x(n,d, correlated = False, means = None, sds = None, highly_correlated = False, seed = 40, standardized = False):
    seed_used = seed
    rng = np.random.default_rng(seed) # Random number generator
    x = []

    if sds is None:
        sds = rng.uniform(0.1,0.5,d)
    if means is None:
        means = [rng.normal(1,sd,1).item() for sd in sds]
    # Put variances in covariance matrix
    cov = np.diag(sds**2)

    if not correlated:
        x = rng.multivariate_normal(means,cov, size = n)
    if correlated:
        # Create off diagonal, triangular matrix
        off_diag = np.triu(rng.uniform(0,2, size = (d,d)))
        cov = cov + off_diag + off_diag.T
        x = rng.multivariate_normal(means,cov, size = n)
    if highly_correlated and d == 3:
        x1 = generate_x(n,1, seed = seed_used)
        x2 = x1 + rng.normal(2,1)
        x3 = x1 + x2
        x = np.concatenate((x1,x2,x3), axis = 1)

    if standardized:
        standardized_x = (x - np.mean(x, axis=0) ) / np.std(x, axis=0)
        x = standardized_x
    return x


In [ ]:
ys = np.concatenate([np.repeat(1, 95), np.repeat(-1,5)])
sum(ys == 1)/len(ys)
# len(ys)
1/100 +0.01

In [ ]:
def generate_y(n,x, betas,method, seed = 40, plot = False, balanced = True):
    y = np.zeros(n)
    prob_y1 = np.zeros(n)
    rng = np.random.default_rng(seed) # Random number generator

    match method:
        case "sinusoidal":
            prob_y1 = [(1/2)*np.sin(2*np.inner(x_t,betas))+1/2 for x_t in x]

            y = [rng.choice([1,-1], size = 1, p = [prob_y1_t, 1-prob_y1_t]).item() for prob_y1_t in prob_y1]

        case "quadratic":
            prob_y1 = [sig(np.inner((x_t**2),betas)) for x_t in x]
            y = [rng.choice([1,-1], size = 1, p = [prob_y1_t, 1-prob_y1_t]).item() for prob_y1_t in prob_y1]

        case "alternating":
            prob_y1 = 0.5 if balanced else 1/n+0.01
            y = rng.choice([1,-1],size = n, p = [prob_y1,1-prob_y1])

        case "sign":
            y = [np.sign(np.inner(x_t,betas)) for x_t in x]

        case "well specified": # Sigmoid function is default
            prob_y1 = [sig(np.inner(x_t,betas)) for x_t in x]
            y = [rng.choice([1,-1], size = 1, p = [prob_y1_t, 1-prob_y1_t]).item() for prob_y1_t in prob_y1]

    # Graphs
    if plot:
        for i in range(x.shape[1]):
            sns.scatterplot(x = x[:,i].flatten(), y = prob_y1)
            plt.title(f"P(Y=1) over x{i+1}")
            plt.show()

    return np.array(y)

## Generate data

In [ ]:
def generate_data(setting, balanced = True, seed = 30, n = default_n, x_means = None, B = 4, a = 1, b = -1, plot_y = False, X_hazan = 1,d = 3, hyperparameter_analysis = False, betas = None, large_betas = False):
    # Allocate space for variables
    x = []
    y = []
    betas = [] if betas is None else betas
    # betas = betas
    rng = np.random.default_rng(seed)
    hazan_py1 = 0
    # Define x means based on balanced or unbalanced setting
    x_means_simulation = [1,-2,0.83] if x_means is None else x_means
    x_means_simulation = [-2,-1.7,1.5] if hyperparameter_analysis else x_means_simulation
    # test_size = round(default_test_ratio*n)
    test_size = default_test_size
    full_n = n+test_size
    av_sigma_hidden = None

    # Generate data according to setup
    match setting:
        case "well specified":
            # Generate data
            x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
            if hyperparameter_analysis:
                betas = [1.4,-3.3,-1.9] if balanced else [1,-0.6,3.9]
                betas = np.array(betas)*10 if large_betas else betas
            elif None in betas or len(betas) == 0:
                betas = [0.4,-0.3,-1.2] if balanced else [-1,-1.7,2.6] # average sigmoid balanced: 0.5, unbalanced : 0.01

            y = generate_y(full_n,x,betas = betas, seed = seed, method = "well specified")

        case "high dimensional" | "hidden x":
            # Define betas and x means to fit high dimensionality
            betas = [2,0.8,-4.2,1.4,0.5,0.2,-1,-0.3,1.2,-0.1,0.6,0.9,-2.4,2.7,-1.4,0.5,-1,4.2,1.5,-0.2,0.4,1.2,3.6,-0.8,-1.7] if balanced else [2,1.8,-3.7,1.4,0.5,0.2,-1,-0.3,2.2,0.1,0.6,1.9,-1.4,2.7,-1.4,0.5,-1,3.2,1.5,-1.5,-2.4,1.2,3.6,-0.8,-1.2]
            x_means_simulation = [-1.6, 2.1, 0.4, 4.3, 0.9, -8.9, 6.9, 3.1,0.4, -1.1, 1.9, 0.3, 2.4, 1.0, -6.3, 0.2,3.1, -2.9, 3.1, 4.6, -0.3, 6.9, 2.4, 1.9, 3.1]
            if hyperparameter_analysis:
                x_means_simulation = [2, -4.2, 1, 2.4, -1.6, -3.8, -5, -4.1,0.2, 6, 1, -3, 4.2, -2.1,5.6, -6,6.3, 4.5, -1.3, 6.2, 2.3, 1.4, 4, -2, 2] if balanced else [2, -4.2, 1, 2.4, -1.6, -3.8, -5, -4.1,0.2, 6, 1, -3, 4.2, -2.1,5.6, -6,6.3, 5.8, -1.3, 6.2, 2.3, 1.4, 4, -2, 2.6]
            # Generate x again for the high dimensional setting
            x = generate_x(full_n,25, means = x_means_simulation, seed = seed)
            y = generate_y(full_n,x,betas = betas, seed = seed, method = "well specified")
            if setting == "hidden x":
                av_sigma_hidden = round(np.mean( [sig(xb) for xb in np.inner(x,betas)]),3)
                betas = betas[:3]
                x = x[:,:3]

        case "quadratic":
            x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
            if hyperparameter_analysis:
                betas = [1.5,1.7,3.9] if balanced else [1,-0.6,3]
            else:
                betas = [-1.44,0.5,-0.8] if balanced else [0.9,-1.1,-1.2]
            y = generate_y(full_n,x,betas = betas, seed = seed, method = "quadratic")
        case "hazan 1-dimensional":
            # B = np.log(full_n) if B_hazan is None else B_hazan
            B = 1.5 if hyperparameter_analysis else 1
            # B = 300
            # epsilon = (B**(2/3))/(n**(2/3))
            epsilon = 0.01
            theta = np.sqrt(epsilon)/B
            hazan_py1 = theta/2 + X_hazan*epsilon/B
            xy = rng.choice([[1- theta/2,a],[theta,b]], size = full_n, p = [theta/2 + X_hazan*epsilon/B, 1- (theta/2 + X_hazan*epsilon/B)])
            x = xy[:,0]
            y = xy[:,1]
            x = x.reshape(full_n,1)

        case "alternating":
            # Generate data
            x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
            if hyperparameter_analysis:
                betas = [1,2.5,0.3]
            elif None in betas or len(betas) == 0:
                betas = [2.4,0.8,-1.2]
            y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "alternating", balanced = balanced)
        case "sinusoidal":
            if not balanced:
                print("Sinusoidal will not be studied in unbalanced case")
            # Generate data

            x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
            if hyperparameter_analysis:
                betas = [1.5,-1,-0.9]
            elif None in betas or len(betas) == 0:
                betas = [2.4,1.8,-0.7] if balanced else [-1.1,0.5,-0.8] # unbalanced average 0.325 on seed 30
            y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "sinusoidal")

        case "sign":
            # Generate data
            x = generate_x(full_n,d, means = x_means_simulation, seed = seed, sds = np.array([1,1,1]))
            if hyperparameter_analysis:
                betas = [-2.3,1.2,-1.4] if balanced else [-1.3,2.6,-3.9]
            elif None in betas or len(betas) == 0:
                betas = [-0.8,-1.4,-2.6] if balanced else [1.3,-3.7,-1.2] # average
            y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "sign")

        case "binomial_x":
            if hyperparameter_analysis:
                x = rng.binomial(n=6,p=0.4,size = (full_n,d))
                betas = [-2.3,1.2,1.1] if balanced else [1.5,-1,-2.9]
            else:
                x = rng.binomial(n = 5, p = 0.7, size = (full_n,d))
                # betas = rng.uniform(size = d,low = -0.5,high = 0.8)
                betas = [1.3,0.3,-1.4] if balanced else [1.5,-1,-2.9]
            # Just to have the correct d later
            x_means_simulation = np.zeros(d)
            y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "well specified")

    if setting == "hazan 1-dimensional":
        print(f"{n} (training) samples were generated using the 1-dimensional setting described by Hazan et al. (2014). With an average P(Y=1) of {hazan_py1}")
    else:
        d = len(x_means_simulation)
        if setting == "sign" or setting == "alternating":

            print(f"{n} (training) samples were generated in the {setting} {"balanced" if balanced else "unbalanced"} setting. Using {d} variable(s), with an average P(Y=1) of {sum(y == 1)/len(y)}")
        else:
            av_sigma = round(np.mean( [sig(xb) for xb in np.inner(x,betas)]),3) if av_sigma_hidden is None else av_sigma_hidden
            av_sin = round(np.mean( [(1/2)*np.sin(5*xb) +1/2 for xb in np.inner(x,betas)]),3)
            print(f"{n} (training) samples were generated in the {setting} {"balanced" if balanced else "unbalanced"} setting. Using {d} variable(s), with an average P(Y=1) of {av_sin if setting == "sinusoidal" else av_sigma}{" using large betas." if large_betas else "."}")

    # Make train/test split

    x_train = x[test_size:]
    y_train = y[test_size:]
    x_test = x[:test_size]
    y_test = y[:test_size]

    return x_train, y_train, x_test, y_test

In [ ]:
a,b,c,d = generate_data("quadratic",balanced = False, n = 100, hyperparameter_analysis=True)

print(sum(b == 1)/len(b))
print(sum(d == 1)/len(d))


## Online Gradient Descent

In [ ]:
def online_gd(data_y, data_x, OTB_method = None, OTB_average_proportion = None):
    """Online gradient descent
    Parameters
    ---------
    n : float
        Size of the dataset
    d : float
        Dimension of the context vector, i.e. number of independent variables
    data_y : ndarray
        Vector with outcome(y) values. Dimensions: n x 1
    data_x: ndarray
        Array with independent variables. Dimensions: n x d
    eta: float
        Learning rate.

    OTB_method: string
        Averaging method, default none which implies the averaging function default which is uniform.
    OTB_average_proportion:
    """

    # Start time
    start_time = datetime.datetime.now()

    # Get dimensions
    n = data_x.shape[0]
    d = data_x.shape[1]

    # Initialize vectors
    betas = np.zeros((n,d)) # Parameter vector of all n rounds.

    loss = np.zeros(n)
    G = 1 # Max gradient
    eta = 1
    gradient = np.zeros(d)
    y_hat = np.zeros(n)

    for t in range(n):
        # Observe context vector for this round
        xt = data_x[t]

        # Update parameter (beta)
        if t == 0:
            beta_tilde = np.zeros(d)
        else:
            beta_tilde = betas[t-1] - eta*gradient

        # Euclidean projection of beta onto the L2 ball
        #betas[t] = project_onto_l2_ball(beta_tilde,B)
        betas[t] = beta_tilde

        # Issue prediction
        y_hat[t] = sig(np.inner(betas[t],xt))

        # Observe true y from the data
        yt = data_y[t]

        # Calculate gradient and sum of gradients
        gradient = -yt*xt*sig(-yt*np.inner(xt,betas[t]))
        # update maximum gradient
        G = np.max([G,np.linalg.norm(gradient)**2])

        # Suffer loss
        loss[t] = np.logaddexp(0,-yt*np.inner(xt,betas[t]))

        # Recalculate eta
        eta = 1 / np.sqrt((t+1) * G) # Here t+1 simply because python initializes t to 0

    # Store endtime
    end_time =  datetime.datetime.now()

    # Get Batch estimate
    betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

    # Get Batch estimate loss
    loss_OTB =  log_loss(betas_OTB,data_x, data_y)

    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss_OTB,
        "Online_Loss" : sum(loss),
        "Batch_estimate": betas_OTB,
        "All_Betas": betas,
        "predictions": y_hat
    }


### Testing ODG

In [ ]:
#ogd = online_gd(n = 40,d = 3,data_y = y,data_x = x,B = 40)

## AIOLI

In [ ]:
class AIOLI:
    def __init__(self):
        self.first_sum  = []
        self.etas_gradients = []
        self.x0 = [] # Initializer for minimizing algorithm
        self.reg_lambda = 0
        self.train_size = 0
        self.d = 0
        self.n = 0

    # Define function to minimize for AIOLI
    @staticmethod # Because I dont use the self
    def parameter_function_AIOLI(b, xt, reg_lambda, first_sum, etas_gradients):
        l_hat = np.inner(b,first_sum) + b @ etas_gradients @ b #  @ to do matrix-vector multiplication
        # Use function np.logaddexp = log(exp(x1) + exp(x2)) to ensure numerical stability, not sure how it works
        result = l_hat + np.logaddexp(0,np.inner(b,xt)) + np.logaddexp(0,np.inner(-b,xt)) + reg_lambda*np.linalg.norm(b)**2
        return result

    def new_point_loss(self,X,Y, OTB_method):
        convergence_new = 0
        beta_hat = np.zeros((self.train_size,self.d))
        first_sum  = np.zeros(self.d)
        etas_gradients = np.zeros((self.d,self.d))
        for t in range(self.train_size):
            # Minimize using data up until t-1
            result = minimize(self.parameter_function_AIOLI, self.x0, method = "l-bfgs-b", args=(X, self.reg_lambda,first_sum, etas_gradients))
            beta_hat[t] = result.x
            convergence_new += result.success
            # Update minimization elements for next round
            first_sum += self.first_sum[t]
            etas_gradients += self.etas_gradients[t]

        # Get OTB
        big_beta = calculate_OTB(beta_hat, OTB_method)

        # Suffer loss
        loss_point = np.logaddexp(0,-Y*np.inner(X,big_beta))

        return loss_point

    def new_data_loss(self,new_data_x,new_data_y, OTB_method: Literal["suffix_averaging", "uniform", "last_iterate"] | None = None):
        start_time = datetime.datetime.now()
        new_losses = [ self.new_point_loss(X,Y, OTB_method) for (X,Y) in zip(new_data_x,new_data_y) ] # Zip pairs the rows of x and y
        end_time = datetime.datetime.now()
        return{
                "runtime" : (end_time - start_time).total_seconds(),
                "loss" : np.mean(new_losses)
            }


    def fit(self, data_y, data_x, B, X, OTB_method = None, OTB_average_proportion = None, minimize_method = "l-bfgs-b", dynamic_B = False):
        """AIOLI
        This is the function to run AIOLI.

        Parameters
        ---------
        data_y : ndarray
            Vector with outcome(y) values. Dimensions: n x 1
        data_x: ndarray
            Array with independent variables. Dimensions: n x d
        B: float
            Radius of the L2 ball constraint for the parameter vector beta
        X: float
            Radius of the L2 ball constraint for the feature vector x"""
        # Record start time
        start_time = datetime.datetime.now()

        # Get data dimensions
        self.n = data_x.shape[0]
        n = self.n
        self.d = data_x.shape[1]
        d = self.d
        self.x0 = np.zeros(d) # Initializer for minimizing algorithm
        self.train_size = n
        # Initialize vectors for storage
        betas = np.zeros((n, d))
        ys = np.zeros(n)
        y_hats = np.zeros(n)
        loss = np.zeros(n)
        etas = np.zeros(n)
        gradients = np.zeros((n, d))
        self.reg_lambda = 1 / (B**2)

        # Keep sums for loss functions
        self.first_sum  = np.zeros((n, d))
        self.etas_gradients = np.zeros((n,d,d))
        first_sum  = np.zeros(d)
        etas_gradients = np.zeros((d,d))

        # Track convergence
        convergence = 0
        minimizer_gradient = 0

        # Run Algorithm
        for t in range(n):
            # Get context vector
            xt = data_x[t]

            # Update beta parameter
            # noinspection PyTypeChecker # This is to avoid the underlining which
            if t == 0: # Initialize to 0 for first round
                betas[t] = np.zeros(d)
            else:
                result = minimize(self.parameter_function_AIOLI,
                                  np.zeros(d),
                                  method = minimize_method,
                                  args=(xt, self.reg_lambda,first_sum, etas_gradients),
                                  options = {"maxiter":100000}
                                  )
                betas[t] = result.x
                convergence += result.success
                minimizer_gradient += np.linalg.norm(result.jac)

            # # Project beta onto set
            # betas[t] = project_onto_l2_ball(beta_tilde,B)
            # betas[t] = self.x0

            # Generate prediction
            y_hats[t] = np.inner(xt,betas[t])

            # Observe true y from data
            ys[t] = data_y[t]

             # Compute gradient
            gradients[t] = -ys[t]*xt*sig(-ys[t]*np.inner(xt,betas[t]))

            # Estimate eta
            etas[t] = np.exp(ys[t]*y_hats[t])/(1+B*X)

            # Suffer loss
            loss[t] = np.logaddexp(0,-ys[t]*np.inner(xt,betas[t]))

            # Compute elements for AIOLI loss function

            fs = gradients[t]*(1 - etas[t]*np.inner(betas[t],gradients[t]))
            eg = (etas[t] / 2) * np.outer(gradients[t], gradients[t])
            self.first_sum[t] = fs
            self.etas_gradients[t] = eg
            first_sum += fs
            etas_gradients += eg
            # Here this is separated and a bit reiterative because else the way the vector is stored messes up with the function. I tried multiple ways and this seemed the best one.

            # Dynamic Hyperparameter tuning
            if dynamic_B:
                B = max(B, np.linalg.norm(betas[t]))

        # for loop ends

        # Get Batch estimate
        betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

        # Get Batch estimate loss
        loss_OTB = log_loss(betas_OTB,data_x, data_y)

        end_time = datetime.datetime.now()

        return{
            "runtime" : (end_time - start_time).total_seconds(),
            "Total_Loss" : loss_OTB,
            "Online_Loss" : sum(loss),
            "Batch_estimate": betas_OTB,
            "All_Betas": betas,
            "convergence" : convergence,
            "minimizer_gradient" : minimizer_gradient
        }



### Testing AIOLI

AIOLI convergence check

In [ ]:
# aioli_model = AIOLI()
# aioli_results_nondynamic = aioli_model.fit(data_y = y_train, data_x = x_train, X = 10, B = 1, minimize_method = "l-bfgs-b", dynamic_B = False)
# aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="uniform")["loss"]

In [ ]:
# aioli_model = AIOLI()
# aioli_results = aioli_model.fit(data_y = y_train, data_x = x_train, X = 10, B = 40, minimize_method = "l-bfgs-b", dynamic_B = True)
# aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="uniform")["loss"]

In [ ]:
# minimization_methods = ["l-bfgs-b",'bfgs',"CG"]
#
# for method in minimization_methods:
#     aioli_model = AIOLI()
#     aioli_results = aioli_model.fit(data_y = y_train, data_x = x_train, X = 10, B = 5, minimize_method = method)
#     print(f"Method: {method}")
#     print(f"convergence: {aioli_results["convergence"]}")
#     print(f"minimizer_gradient: {aioli_results["minimizer_gradient"]}")
#     print(f"Online loss: {aioli_results["Online_Loss"]}")


In [ ]:
# aioli_model.new_data_loss(new_data_x = x_test, new_data_y = y_test, OTB_method = "uniform")

## Ada Grad

In [ ]:
def online_AdaGrad(data_y, data_x, B_vector, OTB_method = None, OTB_average_proportion = None):
    """Online Adaptive Gradient
    Parameters
    ---------
    data_y : ndarray
        Vector with outcome(y) values. Dimensions: n x 1
    data_x: ndarray
        Array with independent variables. Dimensions: n x d
    B_vector: array
        Hyperparemeter, originally represents
    OTB_method: string
        Averaging method, default none which implies the averaging function default which is uniform.
    OTB_average_proportion:
    """

    # Start time
    start_time = datetime.datetime.now()

    # Get dimensions
    n = data_x.shape[0]
    d = data_x.shape[1]

    # Initialize vectors
    betas = np.zeros((n,d)) # Parameter vector of all n rounds.
    loss = np.zeros(n)
    sum_gradient_square = np.zeros(d)

    for t in range(n):
        # Receive x and y
        xt = data_x[t]
        yt = data_y[t]

        # Suffer loss
        loss[t] = np.logaddexp(0,-yt*np.inner(xt,betas[t]))

        # Calculate gradient and sum of gradients
        gradient = -yt*xt*sig(-yt*np.inner(xt,betas[t]))
        sum_gradient_square += gradient**2

        # Output betas
        if t == (n-1):
            pass
        else:
            for i in range(d):
                if gradient[i] != 0:
                    eta_ti = (np.sqrt(2)*B_vector[i])/(2*np.sqrt(sum_gradient_square[i]))
                    betas[t+1,i] = betas[t,i] - eta_ti*gradient[i]
                else:
                    betas[t+1,i] = betas[t,i]

    # Store endtime
    end_time = datetime.datetime.now()

    # Get Batch estimate
    betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)

    # Get Batch estimate loss
    loss_OTB =  log_loss(betas_OTB,data_x, data_y)

    return{
        "runtime" : (end_time - start_time).total_seconds(),
        "Total_Loss" : loss_OTB,
        "Online_Loss" : sum(loss),
        "Batch_estimate": betas_OTB,
        "All_Betas": betas
    }


In [ ]:
# x = generate_x(150,3, means= [2,2,1])
# betas = [2,20,-4.2]
# y = generate_y(150,x,betas = betas)
#
# x_train = x[:100]
# y_train = y[:100]
# x_test = x[100:]
# y_test = y[100:]
#
# standardized_x_train = (x_train - np.mean(x_train, axis=0) ) / np.std(x_train, axis=0)
#x_train = standardized_x_train

In [ ]:
# ag_trial = online_AdaGrad(y_train, x_train, np.repeat(40/13,3))
# ag_trial
# print(log_loss(ag_trial["Batch_estimate"],x_test,y_test))
# ag_u = calculate_OTB(ag_trial["All_Betas"], "uniform")
# ag_s  = calculate_OTB(ag_trial["All_Betas"], "suffix_averaging")
# ag_li  = calculate_OTB(ag_trial["All_Betas"], "last_iterate")
# print(f"{log_loss(ag_u,x_test,y_test)}, {log_loss(ag_s,x_test,y_test)},  {log_loss(ag_li,x_test,y_test)}" )


In [ ]:
# og_trial = online_gd(y_train, x_train, 40)
# og_trial
# log_loss(og_trial["Batch_estimate"],x_test,y_test)

## Logistic Regression

In [ ]:
def classic_logistic_regression(data_x, data_y):
    # Using safe implementation where if data is perfectly separable, then it's just fit minimizing the logistic loss.
    start_time = datetime.datetime.now()

    try:
        logreg = LogisticRegression(penalty=None, fit_intercept=False, max_iter=10000)
        logreg.fit(data_x, data_y)
        betas = logreg.coef_.flatten()

    except ValueError:
        # Single class case: fit manually using scipy minimize on the logistic loss
        def logistic_loss(b):
            scores = data_x @ b
            return np.sum(np.logaddexp(0, -data_y * scores))

        result = minimize(logistic_loss, x0=np.zeros(data_x.shape[1]),
                         options={"maxiter": 10000})
        betas = result.x
        print("Logistic regression was fit manually")


    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.datetime.now()
    return {
        "runtime": (end_time - start_time).total_seconds(),
        "Total_Loss": loss,
        "Batch_estimate": betas
    }

In [ ]:
x_train, y_train, x_test, y_test = generate_data(setting = "well specified", balanced = False, seed = 42, n =50)

In [ ]:
y_train

In [ ]:
b = classic_logistic_regression(x_train,y_train)["Batch_estimate"]
log_loss(b,x_test,y_test)

In [ ]:
# def classic_logistic_regression(data_x,data_y): #Using scikit learn
#     start_time = datetime.datetime.now()
#     # Initialize the model
#     logreg = LogisticRegression(penalty=None, fit_intercept=False)
#
#     # fit the model with data
#     logreg.fit(data_x,data_y)
#     betas = logreg.coef_.flatten() # flatten is To ensure it's a one dimensional array of d elements
#
#     loss = log_loss(betas, data_x, data_y)
#     end_time = datetime.datetime.now()
#     return{
#         "runtime" : (end_time - start_time).total_seconds(),
#         "Total_Loss" : loss,
#         "Batch_estimate": betas
#     }

## Ridge Logistic Regression

In [ ]:
def ridge_logistic_regression(data_x, data_y, reg_lambda=0.1):
    start_time = datetime.datetime.now()

    logreg = LogisticRegression(penalty='l2', C=1/reg_lambda, fit_intercept=False, max_iter=10000)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            logreg.fit(data_x, data_y)
        betas = logreg.coef_.flatten()

    except ValueError:
        def ridge_logistic_loss(b):
            scores = data_x @ b
            return np.sum(np.logaddexp(0, -data_y * scores)) + reg_lambda * np.linalg.norm(b)**2

        result = minimize(ridge_logistic_loss, x0=np.zeros(data_x.shape[1]),
                         options={"maxiter": 10000})
        betas = result.x
        print("Ridge LR was fit manually")

    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.datetime.now()
    return {
        "runtime": (end_time - start_time).total_seconds(),
        "Total_Loss": loss,
        "Batch_estimate": betas
    }

In [ ]:
# def ridge_logistic_regression(data_x,data_y): #Using scikit learn
#     start_time = datetime.datetime.now()
#     # Initialize the model
#     logreg = LogisticRegression(penalty="l2", random_state=1, fit_intercept=False)
#
#     # fit the model with data
#     logreg.fit(data_x,data_y)
#     betas = logreg.coef_.flatten() # flatten is To ensure it's a one dimensional array of d elements
#
#     loss = log_loss(betas, data_x, data_y)
#     end_time = datetime.datetime.now()
#     return{
#         "runtime" : (end_time - start_time).total_seconds(),
#         "Total_Loss" : loss,
#         "Batch_estimate": betas
#     }

## Lasso Logistic Regression

In [ ]:
def lasso_logistic_regression(data_x, data_y, reg_lambda=0.1):
    start_time = datetime.datetime.now()

    logreg = LogisticRegression(penalty='l1', C=1/reg_lambda, fit_intercept=False, max_iter=10000)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            logreg.fit(data_x, data_y)
        betas = logreg.coef_.flatten()

    except ValueError:
        def lasso_logistic_loss(b):
            scores = data_x @ b
            return np.sum(np.logaddexp(0, -data_y * scores)) + reg_lambda * np.abs(np.linalg.norm(b, ord = 1))

        result = minimize(lasso_logistic_loss, x0=np.zeros(data_x.shape[1]),
                         options={"maxiter": 10000})
        betas = result.x
        print("Lasso LR was fit manually")

    loss = log_loss(betas, data_x, data_y)
    end_time = datetime.datetime.now()
    return {
        "runtime": (end_time - start_time).total_seconds(),
        "Total_Loss": loss,
        "Batch_estimate": betas
    }

## Sample Minmax Predictor (SMP)

In [ ]:
class sample_minmax_predictor():
    def __init__(self,data_x, data_y):
        self.data_x = data_x
        self.data_y = data_y
        self.d = 0

    def parameter_function(self,b,x,y = 1):
        # Use y = 1 to get the conditional density for P(Y=1)
        #result = log_loss(b,self.data_x,self.data_y) + log_loss(b,x,y)
        result = np.sum(np.logaddexp(0,-self.data_y*np.inner(self.data_x,b))) + np.logaddexp(0,-y*np.inner(x,b))
        return result

    def get_point_loss(self, new_x,new_y):
        self.d = self.data_x.shape[1]
        x0 = np.zeros(self.d)
        # Get parameter estimate
        beta_hat_y1 = minimize(self.parameter_function, x0=x0, method = "l-bfgs-b", args=(new_x,  1)).x
        beta_hat_y_1 = minimize(self.parameter_function, x0=x0, method = "l-bfgs-b", args=( new_x, -1)).x

        # Get conditional density for P(Y=1)
        f_tilde = sig(1*np.inner(beta_hat_y1,new_x)) / ( sig(np.inner(beta_hat_y1,new_x)*1) + sig(np.inner(beta_hat_y_1,new_x)*(-1)) )

        # Get loss (binary cross entropy)
        loss_point = (new_y == 1)*(-np.log(f_tilde)) + (new_y == -1)*(-np.log(1 - f_tilde)) # Here i have this so that i can actually calculate the loss for both outcomes, in the paper they have it reduced for simplicity, but what they mean is evaluate the density of the observed y, but f_tilde right now is only for y=1, so to calculate both i need to put it in this form instead of simpluy -log (f_tilde)

        # loss_point = np.logaddexp(0,-new_y*np.inner(new_x,beta_hat_y1))

        return loss_point

    def get_new_loss(self, new_data_x, new_data_y):
        start_time = datetime.datetime.now()
        new_losses = [self.get_point_loss(X,Y) for (X,Y) in zip(new_data_x,new_data_y)]
        end_time = datetime.datetime.now()
        return{
            "runtime" : (end_time - start_time).total_seconds(),
            "loss" : np.mean(new_losses),
            "loss_t" : new_losses
        }


### Testing SMP

In [ ]:
# smp = sample_minmax_predictor(x_train, y_train)
# smp.get_new_loss(x_test, y_test)

## Ridge-regularized SMP

In [ ]:
class ridge_sample_minmax_predictor():
    def __init__(self,data_x, data_y, reg_lambda = 0.1):
        self.data_x = data_x
        self.data_y = data_y
        self.d = 0
        self.n = 0
        self.reg_lambda = reg_lambda

    def parameter_function(self,b,x,y = 1):
        # Use y = 1 to get the conditional density for P(Y=1)
        #result = log_loss(b,self.data_x,self.data_y) + log_loss(b,x,y)
        result = (np.sum(np.logaddexp(0,-self.data_y*np.inner(self.data_x,b))) + np.logaddexp(0,-y*np.inner(x,b))) / (self.n + 1) + (self.reg_lambda/2) * np.linalg.norm(b)**2
        return result

    def get_point_loss(self, new_x,new_y):
        self.d = self.data_x.shape[1]
        x0 = np.zeros(self.d)
        # Get parameter estimate
        beta_hat_y1 = minimize(self.parameter_function, x0=x0, method = "l-bfgs-b", args=(new_x,  1)).x
        beta_hat_y_1 = minimize(self.parameter_function, x0=x0, method = "l-bfgs-b", args=( new_x, -1)).x

        # Get conditional density for P(Y=1)
        f_tilde = (sig(1*np.inner(beta_hat_y1,new_x)) * np.exp(1/2 * -self.reg_lambda * np.linalg.norm(beta_hat_y1)**2)) / ( ( sig(np.inner(beta_hat_y1,new_x)*1) * np.exp(1/2 * -self.reg_lambda * np.linalg.norm(beta_hat_y1)**2) )  + ( sig(np.inner(beta_hat_y_1,new_x)*(-1)) * np.exp(1/2 * -self.reg_lambda * np.linalg.norm(beta_hat_y_1)**2) ) )

        # Get loss (binary cross entropy)
        loss_point = (new_y == 1)*(-np.log(f_tilde)) + (new_y == -1)*(-np.log(1 - f_tilde))

        return loss_point

    def get_new_loss(self, new_data_x, new_data_y):
        start_time = datetime.datetime.now()
        self.n = self.data_x.shape[0]
        new_losses = [self.get_point_loss(X,Y) for (X,Y) in zip(new_data_x,new_data_y)]
        end_time = datetime.datetime.now()
        return{
            "runtime" : (end_time - start_time).total_seconds(),
            "loss" : np.mean(new_losses),
            "loss_t" : new_losses
        }


## SMOTE

In [ ]:
class hyperparameter_SMOTE():
    def __init__(self,data_x, data_y ,seed = 40, k = 5, S = 100):
        """ This class performs SMOTE on a dataset. The apply_smote method returns the new data with synthetic samples """
        self.seed = seed
        self.rng = np.random.default_rng(self.seed)
        self.d = 0
        self.k = k
        self.Xmin = []
        self.new_data_y = []
        self.synthetic = [] # array to store synthetic samples
        self.new_i = 0 # variable to keep indices of new synthetic samples. Initialize to 0
        self.data_x = data_x
        self.data_y = data_y
        self.M = 0
        self.S = S
        self.oneclass = False

    def get_min_class_x(self):
        # Get proportion of y = 1
        prop_y1 = np.sum(self.data_y == 1) / len(self.data_y)
        # Get the value of y that is minority
        y_minority = 1 if prop_y1 < 0.5 else -1
        self.oneclass = True if prop_y1 == 1 or prop_y1 == 0 else False
        self.M = np.sum(self.data_y == y_minority)
        # Get only x values for which the outcome is part of the minority
        self.Xmin = self.data_x[self.data_y == y_minority]
        n_new_samples = int((self.S/100)*self.M)
        new_minority_ys = np.full(n_new_samples,y_minority)
        self.new_data_y = np.append(self.data_y,new_minority_ys)
        return True if prop_y1 == 0.5 else False

    def populate(self, s_j, i, nn_indices):
        while s_j != 0:
            # Chose one of the k nearest neighbours of i
            ss = self.rng.integers(0,self.k)
            # Now for every attribute
            for j in range(self.d):
                diff = self.Xmin[nn_indices[ss]][j] - self.Xmin[i][j]
                gap = self.rng.uniform(0,1)
                self.synthetic[self.new_i][j] = self.Xmin[i][j] + gap*diff
            # end for
            self.new_i += 1
            s_j -= 1

    def apply_smote(self):
        """Synthetic Minority Oversampling Technique (SMOTE)
        Parameters
        ---------
        Xmin : array
            Array of minority class samples
        M: int
            Number of minority class samples
        S: int
            Percentual amount of SMOTE
        """

        # Get the Xmin (x values with outcome from minority class)
        no_minority = self.get_min_class_x()

        if no_minority:
            print("Dataset is balanced")
            return self.data_x,self.data_y
        if self.oneclass:
            print("Dataset only has one class")
            return self.data_x,self.data_y
        if self.k > self.M:
            print("k must be smaller than the number of training samples")
            return self.data_x, self.data_y

        # If S is less than 100%, randomize the minority class samples as only a random percent of them will be SMOTEd
        if self.S < 100:
            self.M = (self.S/100)*self.M
            self.S = 100
        # Get number of synthetic samples
        s = int(self.S/100)
        # Get number of attributes
        self.d = self.Xmin.shape[1]
        # Assign correct dimensions to synthetic samples array
        self.synthetic = np.zeros((s*self.M,self.d))

        # Make k nearest neighbors tree
        tree = KDTree(self.Xmin)

        # Get synthetic samples for each datapoint
        for i in range(self.M):
            nn_indices = tree.query(self.Xmin[i:i+1], k = self.k)[1][0]  # First index [1] is to get only the item with the indices, that gives a dataframe of 1 row. Second index [0] is to get that row and simply get an array of the indices
            self.populate(s,i,nn_indices)
        new_data_x = np.concatenate((self.data_x, self.synthetic))
        return new_data_x,self.new_data_y



In [ ]:
class SMOTE():
    def __init__(self,data_x, data_y ,seed = 40, k = 5, S = 100):
        """ This class performs SMOTE on a dataset. The apply_smote method returns the new data with synthetic samples """
        self.seed = seed
        self.rng = np.random.default_rng(self.seed)
        self.d = 0
        self.k = k
        self.Xmin = []
        self.new_data_y = []
        self.synthetic = [] # array to store synthetic samples
        self.new_i = 0 # variable to keep indices of new synthetic samples. Initialize to 0
        self.data_x = data_x
        self.data_y = data_y
        self.M = 0
        self.S = S
        self.oneclass = False

    def get_min_class_x(self):
        # Get proportion of y = 1
        prop_y1 = np.sum(self.data_y == 1) / len(self.data_y)
        # Get the value of y that is minority
        y_minority = 1 if prop_y1 < 0.5 else -1
        self.oneclass = True if prop_y1 == 1 or prop_y1 == 0 else False
        self.M = np.sum(self.data_y == y_minority)
        # Get only x values for which the outcome is part of the minority
        self.Xmin = self.data_x[self.data_y == y_minority]
        n_new_samples = int((self.S/100)*self.M)
        new_minority_ys = np.full(n_new_samples,y_minority)
        self.new_data_y = np.append(self.data_y,new_minority_ys)
        return True if prop_y1 == 0.5 else False

    def populate(self, s_j, i, nn_indices):
        while s_j != 0:
            # Chose one of the k nearest neighbours of i
            ss = self.rng.integers(0,self.k)
            # Now for every attribute
            for j in range(self.d):
                diff = self.Xmin[nn_indices[ss]][j] - self.Xmin[i][j]
                gap = self.rng.uniform(0,1)
                self.synthetic[self.new_i][j] = self.Xmin[i][j] + gap*diff
            # end for
            self.new_i += 1
            s_j -= 1

    def apply_smote(self):
        """Synthetic Minority Oversampling Technique (SMOTE)
        Parameters
        ---------
        Xmin : array
            Array of minority class samples
        M: int
            Number of minority class samples
        S: int
            Percentual amount of SMOTE
        """

        # Get the Xmin (x values with outcome from minority class)
        no_minority = self.get_min_class_x()

        if no_minority:
            print("Dataset is balanced")
            return self.data_x,self.data_y
        if self.oneclass:
            print("Dataset only has one class")
            return self.data_x,self.data_y
        if self.k > self.M:
            self.k = self.M
            print(f"k= {self.k}")

        # If S is less than 100%, randomize the minority class samples as only a random percent of them will be SMOTEd
        if self.S < 100:
            self.M = (self.S/100)*self.M
            self.S = 100
        # Get number of synthetic samples
        s = int(self.S/100)
        # Get number of attributes
        self.d = self.Xmin.shape[1]
        # Assign correct dimensions to synthetic samples array
        self.synthetic = np.zeros((s*self.M,self.d))

        # Make k nearest neighbors tree
        tree = KDTree(self.Xmin)

        # Get synthetic samples for each datapoint
        for i in range(self.M):
            nn_indices = tree.query(self.Xmin[i:i+1], k = self.k)[1][0]  # First index [1] is to get only the item with the indices, that gives a dataframe of 1 row. Second index [0] is to get that row and simply get an array of the indices
            self.populate(s,i,nn_indices)
        new_data_x = np.concatenate((self.data_x, self.synthetic))
        return new_data_x,self.new_data_y



 # Hyperparameter selection

In [ ]:
def hyperparameter_analysis(setting, algorithm, evaluated_parameter,balanced=True, seed=100,n=100, X_hazan=1, d=3, n_runs=20, large_betas = False, large_x = False, other_xmeans = False, x_means = None):

    records = []
    all_losses = []
    values = None
    x_means = [-2,-0.7,-0.5]
    # loss = []
    BX_sequence = np.concatenate((np.arange(1, 4, 1),np.array([np.log(100).round(1)]),np.arange(5,51,15)))
    lambda_sequence = np.concatenate((np.arange(0.1, 1.1, 0.1).round(1),np.arange(2,5,1)))
    for run in range(n_runs):
        run_seed = seed + run

        x_train, y_train, x_test, y_test = generate_data(setting, balanced, run_seed, n, x_means = x_means,
                                                hyperparameter_analysis = True,
                                                X_hazan = X_hazan,
                                                d = d,
                                                large_betas = large_betas
        )
        d_run = x_train.shape[1]

        match evaluated_parameter:
            case "B":
                match algorithm:
                    case "AIOLI":
                        values = BX_sequence
                        loss = np.zeros(len(values))
                        for i in range(len(values)):
                            aioli_model = AIOLI()
                            aioli_model.fit(data_y=y_train, data_x=x_train, X=1, B=values[i],
                                             minimize_method="l-bfgs-b", dynamic_B=False)
                            loss[i] = aioli_model.new_data_loss(
                                new_data_x=x_test, new_data_y=y_test, OTB_method="uniform"
                            )["loss"]
                        for v, l in zip(values, loss):
                            records.append({evaluated_parameter: v, "Risk": l, "run": run})

                    case "ADAGRAD":
                        values = BX_sequence
                        loss = np.zeros((len(values), 3))
                        for i in range(len(values)):
                            adagrad = online_AdaGrad(data_y=y_train, data_x=x_train,
                                                      B_vector=np.repeat(values[i], d_run))
                            ag_u = calculate_OTB(adagrad["All_Betas"], "uniform")
                            ag_s = calculate_OTB(adagrad["All_Betas"], "suffix_averaging")
                            ag_li = calculate_OTB(adagrad["All_Betas"], "last_iterate")
                            loss[i] = [log_loss(ag_u, x_test, y_test),
                                       log_loss(ag_s, x_test, y_test),
                                       log_loss(ag_li, x_test, y_test)]
                        for i, v in enumerate(values):
                            for method, col in zip(["uniform", "suffix", "last_iterate"], range(3)):
                                records.append({evaluated_parameter: v, "Risk": loss[i, col],
                                                 "run": run, "method": method})

            case "X":
                values = BX_sequence
                loss = np.zeros(len(values))
                for i in range(len(values)):
                    aioli_model = AIOLI()
                    aioli_model.fit(data_y=y_train, data_x=x_train, X=values[i], B=np.log(n),
                                     minimize_method="l-bfgs-b")
                    loss[i] = aioli_model.new_data_loss(
                        new_data_x=x_test, new_data_y=y_test, OTB_method="uniform"
                    )["loss"]
                for v, l in zip(values, loss):
                    records.append({evaluated_parameter: v, "Risk": l, "run": run})

            case "S":
                values = np.arange(100, 400, 100)
                loss = np.zeros(len(values))
                for i in range(len(values)):
                    sm = SMOTE(data_x=x_train, data_y=y_train, S=values[i], k =5)
                    smote_x, smote_y = sm.apply_smote()
                    classic_LR_smote = classic_logistic_regression(data_x=smote_x, data_y=smote_y)
                    loss[i] = log_loss(classic_LR_smote["Batch_estimate"], x_test, y_test)
                for v, l in zip(values, loss):
                    records.append({evaluated_parameter: v, "Risk": l, "run": run})

            case "k":
                values = np.arange(1, 10, 1)
                loss = np.zeros(len(values))
                for i in range(len(values)):
                    sm = SMOTE(data_x=x_train, data_y=y_train, k=values[i], S=100)
                    smote_x, smote_y = sm.apply_smote()
                    classic_LR_smote = classic_logistic_regression(data_x=smote_x, data_y=smote_y)
                    loss[i] = log_loss(classic_LR_smote["Batch_estimate"], x_test, y_test)
                for v, l in zip(values, loss):
                    records.append({evaluated_parameter: v, "Risk": l, "run": run})
            case "lambda":
                values = lambda_sequence
                loss = np.zeros(len(values))

                match algorithm:

                    case "Ridge_LR":
                        for i in range(len(values)):
                            ridge_LR = ridge_logistic_regression(data_x=x_train, data_y=y_train, reg_lambda = values[i])
                            loss[i] = log_loss(ridge_LR["Batch_estimate"], x_test, y_test)

                    case "Lasso_LR":
                        for i in range(len(values)):
                            lasso_LR = lasso_logistic_regression(data_x=x_train, data_y=y_train, reg_lambda = values[i])
                            loss[i] = log_loss(lasso_LR["Batch_estimate"], x_test, y_test)

                    case "Ridge_SMP":
                        for i in range(len(values)):
                            ridge_smp = ridge_sample_minmax_predictor(x_train, y_train, reg_lambda= values[i])
                            loss[i] = ridge_smp.get_new_loss(x_test, y_test)["loss"]

                for v, l in zip(values, loss):
                    records.append({evaluated_parameter: v, "Risk": l, "run": run})

        all_losses.append(loss)

    df = pd.DataFrame.from_records(records)

    # Graph
    match algorithm:
        case "AIOLI" | "SMOTE" | "Ridge_LR"|"Lasso_LR"|"Ridge_SMP":

            sns.boxplot(
                data=df,
                x=evaluated_parameter,
                y="Risk",
                fliersize=0,
                showmeans=True,
                meanprops=dict(
                    marker='D',
                    markerfacecolor='white',
                    markeredgecolor='black',
                    markersize=7,
                    markeredgewidth=1.2,
                    zorder=10
                )
            )

            if algorithm == "AIOLI":
                sns.stripplot(
                    data=df,
                    x=evaluated_parameter,
                    y="Risk",
                    color="black",
                    size=4,
                    jitter=True,
                    alpha=0.7
                )

            plt.title(f"Risk over hyperparameter {evaluated_parameter} for {algorithm} in {setting} setting")

            if algorithm == "Ridge_LR" or algorithm == "Lasso_LR" or algorithm == "Ridge_SMP":
                plt.xticks(rotation=90)

            # Build custom legend that includes the mean marker
            ax = plt.gca()
            mean_handle = Line2D(
                [], [],
                marker='D',
                markerfacecolor='white',
                markeredgecolor='black',
                markersize=7,
                linestyle='none',
                label='Expected Risk'
            )
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(
                handles=handles + [mean_handle],
                labels=labels + ['Expected Risk'],
                title="",
                loc="upper right",
                prop={'size': 10}
            )

            plt.show()
            plt.close()
        # AGAGRAD -------------------------------------------------------------------------------------------------------------------------
        case "ADAGRAD":

            fig, axs = plt.subplots(ncols=3, sharey=True)
            for ax, method in zip(axs, ["uniform", "suffix", "last_iterate"]):
                sub = df[df["method"] == method]
                # sns.lineplot(data=sub, x=evaluated_parameter, y="Risk", marker="o", markersize=10,
                #              linewidth=2, color="steelblue", errorbar="sd", ax=ax)
                sns.boxplot(
                    data=sub,
                    x=evaluated_parameter,
                    y="Risk",
                    ax=ax,
                    showmeans=True,
                    meanprops=dict(
                        marker='D',
                        markerfacecolor='white',
                        markeredgecolor='black',
                        markersize=7,
                        markeredgewidth=1.2,
                        zorder=10
                    )
                )

                ax.set_title(method)
                ax.set_xlabel("")
                ax.set_ylabel("")
                ax.tick_params(axis='x', rotation=90, pad=0.2)

            fig.supxlabel("B", size=12)
            fig.supylabel("Risk", size=12)
            fig.suptitle(f"Risk over hyperparameter {evaluated_parameter} for {algorithm} in {setting} setting", size=16)

            # Custom legend for the whole figure (top right)
            mean_handle = Line2D(
                [], [],
                marker='D',
                markerfacecolor='white',
                markeredgecolor='black',
                markersize=7,
                linestyle='none',
                label='Expected Risk'
            )
            fig.legend(
                handles=[mean_handle],
                loc="upper right",
                bbox_to_anchor=(0.85, 0.87),
                prop={'size': 10}
            )

            # plt.xticks(rotation=90)
            plt.show()
            plt.close()

    return df, np.array(all_losses)

In [ ]:
settings = ["well specified", "quadratic", "binomial_x","sinusoidal","sign","hidden x","alternating", "hazan 1-dimensional"]


## Hyperparameter analysis

### AIOLI "B"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "AIOLI", "B", n = 100, n_runs=5)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "AIOLI", "B", n = 100, balanced = False, X_hazan = -1, n_runs=5)

### AIOLI "X"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "AIOLI", "X", n = 100,n_runs = 5)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "AIOLI", "X", n = 100, balanced = False, X_hazan = -1, n_runs = 5)

### Ada Grad "B"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "ADAGRAD", "B", n = 100)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "ADAGRAD", "B", n = 100, balanced = False, X_hazan = -1)

### SMOTE "S"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "SMOTE", "S", n = 100)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "SMOTE", "S", n = 100, balanced = False, X_hazan = -1)

### SMOTE "k"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "SMOTE", "k", n = 100)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "SMOTE", "k", n = 100, balanced = False, X_hazan = -1)

### Ridge LR "lambda"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "Ridge_LR", "lambda", n = 100)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "Ridge_LR", "lambda", n = 100, balanced = False, X_hazan = -1)

### Lasso LR "lambda"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "Lasso_LR", "lambda", n = 100)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "Lasso_LR", "lambda", n = 100, balanced = False, X_hazan = -1)

### Ridge SMP "lambda"

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "Ridge_SMP", "lambda", n = 100)

In [ ]:
for setting in settings:
    hyperparameter_analysis(setting, "Ridge_SMP", "lambda", n = 100, balanced = False, X_hazan = -1)

# Functions for simulations

## Data class for specifying simulation configuration

In [ ]:
default_X = 1
default_B_Adagrad = 3
@dataclass
class SimulationConfig:
    sim_betas: list =  None,
    # Default is well specified
    sim_setting: str =  "well specified"
    sim_balanced: bool = True
    runs: int = default_runs
    n: int = default_n
    d: int = 3
    data_seed: int = 40
    # betas: list = field(default_factory=lambda: [2,0.1,-4.2]) # This is to create a new list every time.
    # x_means: list = field(default_factory=lambda: [2,2,1])
    # These values of betas and x_means guarantee a P(Y=1) = 0.5, hence balanced classes.
    test_ratio: float = default_test_size
    minimize_method: str = "l-bfgs-b"
    y_method: str = "sigmoid"
    X_hazan: int = 1
    large_betas: bool = False
    large_x: bool = False

In [ ]:
# To record algorithm runtime
def time_it(fn):
            """Runs fn(), returns (result, elapsed_seconds)."""
            t0 = datetime.datetime.now()
            result = fn()
            return result, (datetime.datetime.now() - t0).total_seconds()

## Function to run simulations and get results

In [ ]:
def run_simulation(cfg: SimulationConfig = None, B_hazan = None):
    runs = cfg.runs
    n = cfg.n
    d = cfg.d
    minimize_method = cfg.minimize_method
    data_seed = cfg.data_seed
    sim_setting = cfg.sim_setting
    sim_balanced = cfg.sim_balanced
    X_hazan = cfg.X_hazan
    sim_betas = cfg.sim_betas
    large_betas = cfg.large_betas
    large_x = cfg.large_x

    test_set_results = []
    time_results = []
    y_counts = []
    start_time = datetime.datetime.now()

    all_y_trains = {}
    all_y_tests = {}

    for i in range(runs):
        x_train, y_train, x_test, y_test = generate_data(setting = sim_setting,
                                                             balanced = sim_balanced,
                                                             n = n,
                                                             seed = data_seed + i,
                                                             d = d,
                                                             X_hazan = X_hazan,
                                                             betas = sim_betas,
                                                         large_betas = large_betas,
                                                         large_x = large_x)

        y_counts.append({
            "count_1": int(np.sum(y_train == 1)),
            "count_minus1": int(np.sum(y_train == -1))
        })

        all_y_trains[f"run_{i+1}"] = y_train.copy()
        all_y_tests[f"run_{i+1}"] = y_test.copy()

        sm = SMOTE(data_x = x_train, data_y = y_train, S = 100)
        smote_x, smote_y = sm.apply_smote()


        # Fit
        ogd,           t_ogd_fit        = time_it(lambda: online_gd(data_y=y_train, data_x=x_train))
        adagrad,       t_adagrad_fit    = time_it(lambda: online_AdaGrad(data_y=y_train, data_x=x_train, B_vector=np.repeat(default_B_Adagrad, d)))
        aioli_model                     = AIOLI()
        ai,            t_aioli_fit      = time_it(lambda: aioli_model.fit(data_y=y_train, data_x=x_train, X=default_X, B=np.log(n), minimize_method=minimize_method))
        classic_LR,    t_clr_fit        = time_it(lambda: classic_logistic_regression(data_x=x_train, data_y=y_train))
        ridge_LR,      t_ridge_fit      = time_it(lambda: ridge_logistic_regression(data_x=x_train, data_y=y_train))
        lasso_LR,      t_lasso_fit      = time_it(lambda: lasso_logistic_regression(data_x=x_train, data_y=y_train))
        smp,           t_smp_fit        = time_it(lambda: sample_minmax_predictor(x_train, y_train))
        ridge_smp,     t_ridge_smp_fit  = time_it(lambda: ridge_sample_minmax_predictor(x_train, y_train))
        classic_LR_smote, t_smote_fit   = time_it(lambda: classic_logistic_regression(data_x=smote_x, data_y=smote_y))

        #  OTB averaging (counts toward OGD / AdaGrad time)
        OGD_OTB_uniform, t_ogd_u   = time_it(lambda: calculate_OTB(ogd["All_Betas"], "uniform"))
        # OGD_OTB_Suffix,  t_ogd_s   = time_it(lambda: calculate_OTB(ogd["All_Betas"], "suffix_averaging"))
        # OGD_OTB_lastit,  t_ogd_l   = time_it(lambda: calculate_OTB(ogd["All_Betas"], "last_iterate"))

        adagrad_OTB_uniform, t_ada_u = time_it(lambda: calculate_OTB(adagrad["All_Betas"], "uniform"))
        # adagrad_OTB_Suffix,  t_ada_s = time_it(lambda: calculate_OTB(adagrad["All_Betas"], "suffix_averaging"))
        # adagrad_OTB_lastit,  t_ada_l = time_it(lambda: calculate_OTB(adagrad["All_Betas"], "last_iterate"))

        #  Loss on test set
        loss_OGD_uniform_test,  t_ogd_u_loss  = time_it(lambda: log_loss(OGD_OTB_uniform, x_test, y_test))
        # loss_OGD_Suffix_test,   t_ogd_s_loss  = time_it(lambda: log_loss(OGD_OTB_Suffix,  x_test, y_test))
        # loss_OGD_lastit_test,   t_ogd_l_loss  = time_it(lambda: log_loss(OGD_OTB_lastit,  x_test, y_test))

        loss_adagrad_uniform_test, t_ada_u_loss = time_it(lambda: log_loss(adagrad_OTB_uniform, x_test, y_test))
        # loss_adagrad_Suffix_test,  t_ada_s_loss = time_it(lambda: log_loss(adagrad_OTB_Suffix,  x_test, y_test))
        # loss_adagrad_lastit_test,  t_ada_l_loss = time_it(lambda: log_loss(adagrad_OTB_lastit,  x_test, y_test))

        loss_AIOLI_uniform_test, t_aioli_u_loss = time_it(lambda: aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="uniform")["loss"])
        # loss_AIOLI_Suffix_test,  t_aioli_s_loss = time_it(lambda: aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="suffix_averaging")["loss"])
        # loss_AIOLI_lastit_test,  t_aioli_l_loss = time_it(lambda: aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="last_iterate")["loss"])

        loss_LR_test,       t_clr_loss       = time_it(lambda: log_loss(classic_LR["Batch_estimate"],       x_test, y_test))
        loss_LR_smote_test, t_smote_loss     = time_it(lambda: log_loss(classic_LR_smote["Batch_estimate"], x_test, y_test))
        loss_LR_Ridge_test, t_ridge_loss     = time_it(lambda: log_loss(ridge_LR["Batch_estimate"],         x_test, y_test))
        loss_LR_Lasso_test, t_lasso_loss     = time_it(lambda: log_loss(lasso_LR["Batch_estimate"],         x_test, y_test))

        smp_result,       t_smp_loss      = time_it(lambda: smp.get_new_loss(x_test, y_test))
        ridge_smp_result, t_ridge_smp_loss = time_it(lambda: ridge_smp.get_new_loss(x_test, y_test))

        #  Collate results
        algorithms_test = {
            "Classical_LR":         {"loss": loss_LR_test,              "time_fit": t_clr_fit,       "time_loss": t_clr_loss},
            "Classical_LR_SMOTE":   {"loss": loss_LR_smote_test,        "time_fit": t_smote_fit,     "time_loss": t_smote_loss},
            "Ridge_LR":             {"loss": loss_LR_Ridge_test,         "time_fit": t_ridge_fit,     "time_loss": t_ridge_loss},
            "Lasso_LR":             {"loss": loss_LR_Lasso_test,         "time_fit": t_lasso_fit,     "time_loss": t_lasso_loss},
            "OGD_uniform":          {"loss": loss_OGD_uniform_test,      "time_fit": t_ogd_fit,       "time_loss": t_ogd_u + t_ogd_u_loss},
            # "OGD_Suffix":           {"loss": loss_OGD_Suffix_test,       "time_fit": t_ogd_fit,       "time_loss": t_ogd_s + t_ogd_s_loss},
            # "OGD_Last_iterate":     {"loss": loss_OGD_lastit_test,       "time_fit": t_ogd_fit,       "time_loss": t_ogd_l + t_ogd_l_loss},
            "AdaGrad_uniform":      {"loss": loss_adagrad_uniform_test,  "time_fit": t_adagrad_fit,   "time_loss": t_ada_u + t_ada_u_loss},
            # "AdaGrad_Suffix":       {"loss": loss_adagrad_Suffix_test,   "time_fit": t_adagrad_fit,   "time_loss": t_ada_s + t_ada_s_loss},
            # "AdaGrad_Last_iterate": {"loss": loss_adagrad_lastit_test,   "time_fit": t_adagrad_fit,   "time_loss": t_ada_l + t_ada_l_loss},
            "AIOLI_uniform":        {"loss": loss_AIOLI_uniform_test,    "time_fit": t_aioli_fit,     "time_loss": t_aioli_u_loss},
            # "AIOLI_Suffix":         {"loss": loss_AIOLI_Suffix_test,     "time_fit": t_aioli_fit,     "time_loss": t_aioli_s_loss},
            # "AIOLI_lastit":         {"loss": loss_AIOLI_lastit_test,     "time_fit": t_aioli_fit,     "time_loss": t_aioli_l_loss},
            "SMP":                  {"loss": smp_result["loss"],         "time_fit": t_smp_fit,       "time_loss": t_smp_loss},
            "Ridge_SMP":            {"loss": ridge_smp_result["loss"],   "time_fit": t_ridge_smp_fit, "time_loss": t_ridge_smp_loss},
        }

        for name, metrics in algorithms_test.items():
            row = {"run": i + 1, "algorithm": name, **metrics}
            row["time_total"] = row["time_fit"] + row["time_loss"]
            test_set_results.append(row)

        print(f"Run {i+1} finished")

    endtime = datetime.datetime.now()
    total_time = (endtime - start_time).total_seconds()

    results_df = pd.DataFrame(test_set_results)

    return {
        "test_results":    results_df[["run", "algorithm", "loss"]],
        "time_results":    results_df[["run", "algorithm", "time_fit", "time_loss", "time_total"]],
        "y_counts_train":  pd.DataFrame(y_counts),
        "y_train_df":      pd.DataFrame(all_y_trains),
        "y_test_df":       pd.DataFrame(all_y_tests),
        "total_time":      total_time,
    }

## High dimensional simulations for selected algorithms

In [ ]:
def HD_run_simulation(cfg: SimulationConfig = None):
    runs = cfg.runs
    n = cfg.n
    d = cfg.d
    minimize_method = cfg.minimize_method
    data_seed = cfg.data_seed
    sim_setting = cfg.sim_setting
    sim_balanced = cfg.sim_balanced
    sim_betas = cfg.sim_betas

    test_set_results = []
    time_results = []
    y_counts = []
    start_time = datetime.datetime.now()
    total_time = 0

    all_y_trains = {}
    all_y_tests = {}

    for i in range(runs):
        x_train, y_train, x_test, y_test = generate_data(setting = sim_setting,
                                                             balanced = sim_balanced,
                                                             n = n,
                                                             seed = data_seed + i,
                                                             d = d,
                                                             betas = sim_betas)

        y_counts.append({
            "count_1": int(np.sum(y_train == 1)),
            "count_minus1": int(np.sum(y_train == -1))
        })

        all_y_trains[f"run_{i+1}"] = y_train.copy()
        all_y_tests[f"run_{i+1}"] = y_test.copy()

        sm = SMOTE(data_x = x_train, data_y = y_train, S = 100)
        smote_x, smote_y = sm.apply_smote()

        # Fit

        aioli_model                     = AIOLI()
        ai,            t_aioli_fit      = time_it(lambda: aioli_model.fit(data_y=y_train, data_x=x_train, X=default_X, B=np.log(n), minimize_method=minimize_method))
        classic_LR,    t_clr_fit        = time_it(lambda: classic_logistic_regression(data_x=x_train, data_y=y_train))
        ridge_LR,      t_ridge_fit      = time_it(lambda: ridge_logistic_regression(data_x=x_train, data_y=y_train))
        lasso_LR,      t_lasso_fit      = time_it(lambda: lasso_logistic_regression(data_x=x_train, data_y=y_train))
        smp,           t_smp_fit        = time_it(lambda: sample_minmax_predictor(x_train, y_train))
        ridge_smp,     t_ridge_smp_fit  = time_it(lambda: ridge_sample_minmax_predictor(x_train, y_train))
        classic_LR_smote, t_smote_fit   = time_it(lambda: classic_logistic_regression(data_x=smote_x, data_y=smote_y))

        loss_AIOLI_uniform_test, t_aioli_u_loss = time_it(lambda: aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="uniform")["loss"])

        loss_LR_test,       t_clr_loss       = time_it(lambda: log_loss(classic_LR["Batch_estimate"],       x_test, y_test))
        loss_LR_smote_test, t_smote_loss     = time_it(lambda: log_loss(classic_LR_smote["Batch_estimate"], x_test, y_test))
        loss_LR_Ridge_test, t_ridge_loss     = time_it(lambda: log_loss(ridge_LR["Batch_estimate"],         x_test, y_test))
        loss_LR_Lasso_test, t_lasso_loss     = time_it(lambda: log_loss(lasso_LR["Batch_estimate"],         x_test, y_test))

        smp_result,       t_smp_loss      = time_it(lambda: smp.get_new_loss(x_test, y_test))
        ridge_smp_result, t_ridge_smp_loss = time_it(lambda: ridge_smp.get_new_loss(x_test, y_test))

        #  Collate results
        algorithms_test = {
            "Classical_LR":         {"loss": loss_LR_test,              "time_fit": t_clr_fit,       "time_loss": t_clr_loss},
            "Classical_LR_SMOTE":   {"loss": loss_LR_smote_test,        "time_fit": t_smote_fit,     "time_loss": t_smote_loss},
            "Ridge_LR":             {"loss": loss_LR_Ridge_test,         "time_fit": t_ridge_fit,     "time_loss": t_ridge_loss},
            "Lasso_LR":             {"loss": loss_LR_Lasso_test,         "time_fit": t_lasso_fit,     "time_loss": t_lasso_loss},
            "AIOLI_uniform":        {"loss": loss_AIOLI_uniform_test,    "time_fit": t_aioli_fit,     "time_loss": t_aioli_u_loss},
            "SMP":                  {"loss": smp_result["loss"],         "time_fit": t_smp_fit,       "time_loss": t_smp_loss},
            "Ridge_SMP":            {"loss": ridge_smp_result["loss"],   "time_fit": t_ridge_smp_fit, "time_loss": t_ridge_smp_loss},
        }

        for name, metrics in algorithms_test.items():
            row = {"run": i + 1, "algorithm": name, **metrics}
            row["time_total"] = row["time_fit"] + row["time_loss"]
            test_set_results.append(row)

        print(f"Run {i+1} finished")

    endtime = datetime.datetime.now()
    total_time += (endtime - start_time).total_seconds()

    results_df = pd.DataFrame(test_set_results)

    return {
        "test_results":    results_df[["run", "algorithm", "loss"]],
        "time_results":    results_df[["run", "algorithm", "time_fit", "time_loss", "time_total"]],
        "y_counts_train":  pd.DataFrame(y_counts),
        "y_train_df":      pd.DataFrame(all_y_trains),
        "y_test_df":       pd.DataFrame(all_y_tests),
        "total_time":      total_time,
    }

## Simulations changing n

In [ ]:
def simulations_changing_n(n_values: list, base_config: SimulationConfig, filename, B_hazan = None, high_dimensional_analysis = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Run simulations for multiple sample sizes and aggregate results.

    Args:
        n_values: List of sample sizes to simulate
        base_config: Base SimulationConfig object (will copy and modify n for each run)

    Returns:
        summary_df: DataFrame with columns: n, algorithm, avg_loss, std_loss, num_runs
        raw_df: DataFrame with columns: n, algorithm, loss (one row per run)
    """
    all_summaries = []
    all_raw = []
    total_time = 0
    results_dict = None

    for n in n_values:
        print(f"="*50)
        print(f"Running simulations for n = {n}\n")
        print(f"="*50)
        # Create a copy of the config with the new n value
        config = base_config
        config.n = n  # Modify n in place

        # Run the simulation
        if high_dimensional_analysis:
            results_dict = HD_run_simulation(config)
        else:
            results_dict = run_simulation(config)

        # Extract test results
        test_results_df = results_dict["test_results"]
        total_time += results_dict["total_time"]

        # Add n column to raw data
        raw_with_n = test_results_df.copy()
        raw_with_n['n'] = n
        all_raw.append(raw_with_n)

        # Calculate average loss and standard deviation for each algorithm
        summary = test_results_df.groupby('algorithm').agg({
            'loss': ['mean', 'std', 'count']
        }).reset_index()


        # Flatten column names
        summary.columns = ['algorithm', 'avg_loss', 'std_loss', 'num_runs']

        summary['se_loss'] = summary['std_loss'] / np.sqrt(summary['num_runs'])
        summary = summary.drop(columns=["std_loss"])

        # Add n value
        summary['n'] = n

        # Reorder columns
        summary = summary[['n', 'algorithm', 'avg_loss', 'se_loss', 'num_runs']]

        all_summaries.append(summary)

        print(f"Completed simulations for n = {n}")

        # Save to csv
        temp_summary_df = pd.concat(all_summaries, ignore_index=True)
        temp_summary_df.to_csv(f"{filename}.csv")

    # Combine all results
    summary_df = pd.concat(all_summaries, ignore_index=True)
    raw_df = pd.concat(all_raw, ignore_index=True)
    print(f"Total time: {total_time} seconds")
    return summary_df, raw_df

In [ ]:
def plot_simulation_results(
    raw_data: pd.DataFrame,
    algorithms: list = None,
    figsize: tuple = (12, 8),
    title: str = 'Algorithm Performance vs Sample Size',
    xlabel: str = 'Sample Size (n)',
    ylabel: str = 'Risk',
    box_alpha: float = 0.6,
    show_plot: bool = True
) -> plt.Figure:
    """
    Plot box plots of loss distributions per algorithm across different sample sizes.

    Args:
        raw_data: DataFrame with columns: algorithm, n, loss (one row per run)
        algorithms: List of algorithm names to plot (None = all)
        figsize: Figure size (width, height)
        title: Plot title
        xlabel: X-axis label
        ylabel: Y-axis label
        box_alpha: Transparency of box fill (0 = transparent, 1 = opaque)
        show_plot: Whether to display the plot

    Returns:
        None
    """
    # Filter algorithms if specified
    raw_plot = raw_data.copy()
    if algorithms is not None:
        raw_plot = raw_plot[raw_plot['algorithm'].isin(algorithms)]

    if raw_plot.empty:
        print("No data to plot.")
        return None

    # Ensure n is treated as a categorical axis so boxes are evenly spaced
    raw_plot["n"] = raw_plot["n"].astype(str)

    fig, ax = plt.subplots(figsize=figsize)
    sns.set_style("whitegrid")

    sns.boxplot(
        data=raw_plot,
        x="n",
        y="loss",
        hue="algorithm",
        palette="bright",
        linewidth=1.2,
        flierprops=dict(marker='o', markersize=4, alpha=0.5, linestyle='none'),
        showmeans = True,
        meanprops=dict(
        marker="D",
        markerfacecolor="white",
        markeredgecolor="black",
        markeredgewidth=1.2,
        zorder=10,
        ),
        ax=ax
    )

    # Apply alpha to box fills post-hoc
    for patch in ax.patches:
        r, g, b, _ = patch.get_facecolor()
        patch.set_facecolor((r, g, b, box_alpha))

    # Formatting
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=12, fontweight='bold')
    ax.legend(title="Algorithm", loc='best', prop={'size': 10})
    ax.grid(True, alpha=0.3)


    mean_handle = Line2D([], [], marker='D', markerfacecolor='white', markeredgecolor='black',
                          markersize=7, linestyle='none', label='Expected Risk')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [mean_handle], labels=labels + ['Expected Risk'],
              title="Algorithm", loc="best", prop={'size': 10})

    plt.tight_layout()

    if show_plot:
        plt.show()

    return None

## Function for printing results

In [ ]:
def get_variable_name(var, namespace=None):
    if namespace is None:
        namespace = globals()
    for name, value in namespace.items():
        if value is var:
            return name
    return None

In [ ]:
def print_results(results, n=default_n, plot=False, selected_algorithms=None, runs=default_runs, box_alpha=0.6, summary_object = False):
    # Save to csv
    df_name = get_variable_name(results)
    results["test_results"].to_csv(f"{df_name}.csv")

    #  Loss summary 
    summary_test_average = results["test_results"].groupby("algorithm")[["loss"]].mean()
    summary_test_se      = results["test_results"].groupby("algorithm")[["loss"]].std(ddof=1) / np.sqrt(runs)

    summary = pd.concat([summary_test_average, summary_test_se], axis=1)
    summary.columns = ["Average_Loss", "SE_Loss"]

    #  Time summary 
    time_avg = results["time_results"].groupby("algorithm")[["time_fit", "time_loss", "time_total"]].mean()
    time_se  = results["time_results"].groupby("algorithm")[["time_total"]].std(ddof=1) / np.sqrt(runs)

    time_summary = pd.concat([time_avg, time_se], axis=1)
    time_summary.columns = [
        "Avg_Time_Fit", "Avg_Time_Loss", "Avg_Time_Total", "SE_Time_Total"
    ]

    #  Print 
    print(f"Total time: {str(datetime.timedelta(seconds=results['total_time']))}")

    print("\n Loss Results ")
    display(summary)

    print("\n Time Results (seconds) ")
    display(time_summary)

    if not summary_object:
        return
    else:
        return summary

    #  Boxplot 
    if not plot:
        return

    raw_data = results["test_results"].copy()
    if selected_algorithms is not None:
        raw_data = raw_data[raw_data["algorithm"].isin(selected_algorithms)]

    if raw_data.empty:
        print("No data to plot.")
        return

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.set_style("whitegrid")

    sns.boxplot(
        data=raw_data,
        x="algorithm",
        y="loss",
        palette="bright",
        linewidth=1.2,
        flierprops=dict(marker='o', markersize=4, alpha=0.5, linestyle='none'),
        ax=ax
    )

    # Apply alpha to box fills post-hoc
    for patch in ax.patches:
        r, g, b, _ = patch.get_facecolor()
        patch.set_facecolor((r, g, b, box_alpha))

    # Aesthetics
    ax.set_xlabel("Algorithm", fontsize=12, fontweight="bold")
    ax.set_ylabel("Risk", fontsize=12, fontweight="bold")
    ax.set_title("Risk distribution across runs", fontsize=14, fontweight="bold")
    ax.grid(axis='y', linestyle='--', alpha=0.4)

    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Plot results (Function for graphing loss and Y=1 counts over runs)

In [ ]:
def plot_results(results, n = default_n, plot = False):
    test_results = results["test_results"]

    # Boxplots of y counts
    if plot:
        sns.boxplot(data=results["y_counts_train"][["count_1", "count_minus1"]])
        plt.title("Distribution of Y counts across runs")
        plt.ylim(-10, n)
        plt.show()
        # Line plot of Loss over rounds
        sns.lineplot(x = test_results["run"], y = test_results["loss"], hue = test_results["algorithm"]).set_title("Loss over rounds")
        plt.show()
    else:
        pass

    # Distribution of y values
    print("Train set")
    print(results["y_train_df"].apply(pd.Series.value_counts))
    print("Test set")
    print(results["y_test_df"].apply(pd.Series.value_counts))


## Compare settings

In [ ]:
def compare_settings(
    settings_dict: dict,
    selected_algorithms: list = None,
    x_labels: dict = None,
    figsize: tuple = (12, 6),
    title: str = 'Algorithm Performance Across Balanced Settings',
    xlabel: str = 'Setting',
    ylabel: str = 'Risk',
    box_alpha: float = 0.6,
    show_plot: bool = True,
    return_data: bool = False
):
    """
    Combine results from multiple settings and create a box plot of loss
    per algorithm across runs.

    Parameters:
        settings_dict : dict
            Keys are setting names (str), values are results dictionaries from run_simulation.
        selected_algorithms : list, optional
            List of algorithm names to include. If None, all are plotted.
        x_labels : dict, optional
            Mapping from setting key to display label on the x-axis.
            e.g. {"s1": "Well Specified", "s2": "High Dimensional"}
            If None, the setting keys are used as-is.
        figsize : tuple
            Figure size.
        title, xlabel, ylabel : str
            Plot labels.
        box_alpha : float
            Transparency of the box fill (0 = transparent, 1 = opaque).
        show_plot : bool
            Whether to display the plot.
        return_data : bool
            If True, return the combined raw DataFrame.

    Returns:
        raw_df : pd.DataFrame (optional)
            Combined raw data with columns: setting, algorithm, loss.
    """
    # Collect raw data from all settings
    all_raw = []
    for setting, res in settings_dict.items():
        df = res["test_results"].copy()
        df["setting"] = setting
        all_raw.append(df)
    combined_raw = pd.concat(all_raw, ignore_index=True)

    # Remap setting keys to display labels if provided
    if x_labels is not None:
        combined_raw["setting"] = combined_raw["setting"].map(x_labels).fillna(combined_raw["setting"])

    # Filter algorithms if needed
    if selected_algorithms is not None:
        combined_raw = combined_raw[combined_raw["algorithm"].isin(selected_algorithms)]

    if combined_raw.empty:
        print("No data to plot.")
        return

    plt.figure(figsize=figsize)
    sns.set_style("whitegrid")

    ax = sns.boxplot(
        data=combined_raw,
        x="setting",
        y="loss",
        hue="algorithm",
        palette="bright",
        linewidth=1.2,
        flierprops=dict(marker='o', markersize=4, alpha=0.5, linestyle='none'),
        showmeans=True,
        meanprops=dict(
            marker='D',
            markerfacecolor='white',
            markeredgecolor='black',
            markersize=7,
            markeredgewidth=1.2,
            zorder=10,
        ),
    )

    # Apply alpha to box fills after
    for patch in ax.patches:
        r, g, b, _ = patch.get_facecolor()
        patch.set_facecolor((r, g, b, box_alpha))

    # Formatting
    ax.set_title(title, fontsize=16, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=14, fontweight="bold")
    ax.set_ylabel(ylabel, fontsize=12, fontweight="bold")

    # Legend: algorithm colors + a manual "ER" entry explaining the  marker

    mean_handle = Line2D([], [], marker='D', markerfacecolor='white', markeredgecolor='black',
                          markersize=7, linestyle='none', label='Expected Risk')
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [mean_handle], labels=labels + ['Expected Risk'],
              title="Algorithm", loc="best", prop={'size': 10})

    plt.xticks(rotation=0)
    plt.tight_layout()

    if show_plot:
        plt.show()

    if return_data:
        return combined_raw
    else:
        return None

# Experiments

## Hazan setting

In [ ]:
hazan = SimulationConfig(
    sim_setting = "hazan 1-dimensional"
)

hazan_result = run_simulation(hazan)
print_results(hazan_result)

In [ ]:
hazan_result["test_results"]

In [ ]:
plot_results(hazan_result)

In [ ]:
hazan_Xminus1 = SimulationConfig(
    sim_setting = "hazan 1-dimensional",
    n = 100,
    runs = 10,
    X_hazan = -1
)

hazan_Xminus1_result = run_simulation(hazan_Xminus1)
print_results(hazan_Xminus1_result)

In [ ]:
hazan_Xminus1_result["test_results"]

In [ ]:
plot_results(hazan_Xminus1_result)

In [ ]:
settings_results_hazan = {
    "Hazan": hazan_result,
    "Hazan_Xminus1": hazan_Xminus1_result
}

In [ ]:
compare_settings(settings_results_hazan, title = "Algorithms Performance in Adversarial Distributions by Hazan et al. 2014", selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Ridge_LR","Lasso_LR","OGD_uniform","AdaGrad_uniform","AIOLI_uniform","SMP","Ridge_SMP"])

In [ ]:

compare_settings(
    settings_results_hazan,
    selected_algorithms=["Classical_LR", "SMP", "AIOLI_uniform", "Ridge_LR"], title = "Algorithms Performance in Adversarial Distributions by Hazan et al. 2014"
)

## Balanced

In [ ]:
#Binomial
binomial = SimulationConfig(
    sim_setting = "binomial_x",
    n = 100
)

binomial_result = run_simulation(binomial)
print_results(binomial_result)

In [ ]:
plot_results(binomial_result)

In [ ]:
# Well Specified
well_specified = SimulationConfig(
    sim_setting = "well specified",
    d = 3
)

well_specified_result = run_simulation(well_specified)
print_results(well_specified_result)

In [ ]:
plot_results(well_specified_result)

In [ ]:
quadratic = SimulationConfig(
    sim_setting = "quadratic",
    d = 3
)

quadratic_result = run_simulation(quadratic)
print_results(quadratic_result)

In [ ]:
plot_results(quadratic_result)

In [ ]:
# Sinusoidal
sinusoidal = SimulationConfig(
    sim_setting = "sinusoidal",
    n = 100
)

sinusoidal_result = run_simulation(sinusoidal)
print_results(sinusoidal_result)

In [ ]:
# Sign
sign = SimulationConfig(
    sim_setting = "sign",
    n = 100,
    d = 3
)

sign_result = run_simulation(sign)
print_results(sign_result)

In [ ]:
# Hidden X
hidden_x = SimulationConfig(
    sim_setting = "hidden x",
    n = 100,
    d = 3
)

hidden_x_result = run_simulation(hidden_x)
print_results(hidden_x_result)

In [ ]:
# Alternating
alternating = SimulationConfig(
    sim_setting = "alternating",
    n = 100,
    d = 3
)

alternating_result = run_simulation(alternating)
print_results(alternating_result)

In [ ]:
# High Dimensional
high_dimensional = SimulationConfig(
    sim_setting = "high dimensional",
    n = 20,
    d = 25
)

In [ ]:
high_dimensional_result = run_simulation(high_dimensional)
print_results(high_dimensional_result)

In [ ]:
# Understanding Difference in risk of LR in balanced vs unbalanced high dimensional
# a,b,c,d = generate_data("high dimensional", n =20, seed = 40, balanced=False)
# LR = classic_logistic_regression(a,b)
# log_loss(LR["Batch_estimate"], c,d)
# LR["Batch_estimate"]
#
# betas_n20 = LR["Batch_estimate"]
#
# a_n50,b_n50,c_n50,d_n50 = generate_data("high dimensional", n =50, seed = 40, balanced = False)
# LR_n50 = classic_logistic_regression(a_n50,b_n50)
#
# betas_n50 = LR_n50["Batch_estimate"]
#
# print(betas_n20)
# print(betas_n50)

In [ ]:
plot_results(high_dimensional_result, n = 20)

In [ ]:
# High Dimensionality
high_dimensionality = SimulationConfig(
    sim_setting = "high dimensional",
    n = 50,
    d = 25
)

high_dimensionality_result = run_simulation(high_dimensionality)
print_results(high_dimensionality_result)

In [ ]:
plot_results(high_dimensionality_result, n = 50)

In [ ]:
# Graph groups
settings_results_HD = {
    "High Dimensional": high_dimensional_result,
    "High Dimensionality": high_dimensionality_result

}

settings_results_misspecified = {
    "Sinusoidal" : sinusoidal_result,
    "Sign" : sign_result,
    "Hidden X" : hidden_x_result,
    "Alternating" : alternating_result,
}

settings_results_wellspecified = {
    "Well specified": well_specified_result,
    "Quadratic": quadratic_result,
    "Binomial X": binomial_result
}

### Plot results for Averaging Strategies

In [ ]:
# compare_settings(
#     settings_results_HD,
#     title = "Algorithm Performance for Balanced High Dimensional settings", selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
#                                                                                                      "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
#                                                                                                      "AIOLI_Suffix","AIOLI_lastit"],
#     x_labels={"High Dimensional": "High Dimensional (n = 20, d = 25)", "High Dimensionality": "High Dimensionality (n = 50, d = 25)"}
# )

In [ ]:
# compare_settings(
#     settings_results_wellspecified, selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
#                                                                                                      "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
#                                                                                                      "AIOLI_Suffix","AIOLI_lastit"]
# )

In [ ]:
# compare_settings(
#     settings_results_misspecified, selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
#                                                                                                      "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
#                                                                                                      "AIOLI_Suffix","AIOLI_lastit"]
# )

### Plot risk results

In [ ]:
compare_settings(
    settings_results_wellspecified, selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Ridge_LR","OGD_uniform","AdaGrad_uniform","AIOLI_uniform","SMP","Ridge_SMP","Lasso_LR"]
)

In [ ]:
compare_settings(
    settings_results_HD,
    title = "Algorithm Performance for Balanced High Dimensional settings", selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Ridge_LR","Lasso_LR","OGD_uniform","AdaGrad_uniform","AIOLI_uniform","SMP","Ridge_SMP"],
    x_labels={"High Dimensional": "High Dimensional (n = 20, d = 25)", "High Dimensionality": "High Dimensionality (n = 50, d = 25)"}
)

In [ ]:
# All except LR
compare_settings(
    settings_results_HD,
    title="Algorithm Performance for Balanced High Dimensional settings",
    selected_algorithms=["Ridge_LR", "Lasso_LR", "OGD_uniform", "AdaGrad_uniform",
                         "AIOLI_uniform", "SMP", "Ridge_SMP"],
    x_labels={"High Dimensional": "High Dimensional (n = 20, d = 25)",
              "High Dimensionality": "High Dimensionality (n = 50, d = 25)"}
)

In [ ]:
compare_settings(
    settings_results_misspecified, selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Ridge_LR","OGD_uniform","AdaGrad_uniform","AIOLI_uniform","SMP","Ridge_SMP","Lasso_LR"]
)

## Unbalanced

### Experiments

In [ ]:
# Binomial
binomial_unbalanced = SimulationConfig(
    sim_setting = "binomial_x",
    n = 100,
    sim_balanced= False
)

binomial_unbalanced_result = run_simulation(binomial_unbalanced)
print_results(binomial_unbalanced_result)

In [ ]:
plot_results(binomial_unbalanced_result, n = 100)

In [ ]:
# Well specified
well_specified_unbalanced = SimulationConfig(
    sim_setting = "well specified",
    sim_balanced= False
)

well_specified_unbalanced_result = run_simulation(well_specified_unbalanced)
print_results(well_specified_unbalanced_result)

In [ ]:
plot_results(well_specified_unbalanced_result)

In [ ]:
#Quadratic
quadratic_unbalanced = SimulationConfig(
    sim_setting = "quadratic",
    d = 3,
    sim_balanced= False
)

quadratic_unbalanced_result = run_simulation(quadratic_unbalanced)
print_results(quadratic_unbalanced_result)

In [ ]:
plot_results(quadratic_unbalanced_result)

In [ ]:
# High Dimensional
high_dimensional_unbalanced = SimulationConfig(
    sim_setting = "high dimensional",
    n = 20,
    d = 25,
    sim_balanced= False
)

high_dimensional_unbalanced_result = run_simulation(high_dimensional_unbalanced)
print_results(high_dimensional_unbalanced_result)


In [ ]:
plot_results(high_dimensional_unbalanced_result, n = 20)

In [ ]:
# High dimensionality
high_dimensionality_unbalanced = SimulationConfig(
    sim_setting = "high dimensional",
    n = 50,
    d = 25,
    sim_balanced = False
)

high_dimensionality_unbalanced_result = run_simulation(high_dimensionality_unbalanced)
print_results(high_dimensionality_unbalanced_result)


In [ ]:
plot_results(high_dimensionality_unbalanced_result, n =50)

In [ ]:
# Sign
sign_unbalanced = SimulationConfig(
    sim_setting = "sign",
    n = 100,
    sim_balanced = False
)

sign_unbalanced_result = run_simulation(sign_unbalanced)
print_results(sign_unbalanced_result)

In [ ]:
plot_results(sign_unbalanced_result)

In [ ]:
# Alternating
alternating_unbalanced = SimulationConfig(
    sim_setting = "alternating",
    n = 100,
    sim_balanced = False
)

alternating_unbalanced_result = run_simulation(alternating_unbalanced)
print_results(alternating_unbalanced_result)

In [ ]:
plot_results(alternating_unbalanced_result)

In [ ]:
# Hidden X
hidden_x_unbalanced = SimulationConfig(
    sim_setting = "hidden x",
    n = 100,
    sim_balanced = False
)

hidden_x_unbalanced_result = run_simulation(hidden_x_unbalanced)
print_results(hidden_x_unbalanced_result)

In [ ]:
plot_results(hidden_x_unbalanced_result)

In [ ]:
#Graph groups
settings_unbalanced_results_wellspecified = {
    "Well specified": well_specified_unbalanced_result,
    "Quadratic": quadratic_unbalanced_result,
    "Binomial X": binomial_unbalanced_result
}

settings_unbalanced_results_misspecified = {
    "Sign" : sign_unbalanced_result,
    "Hidden X" : hidden_x_unbalanced_result,
    "Alternating":alternating_unbalanced_result
}

settings_unbalanced_results_HD = {
    "High Dimensional": high_dimensional_unbalanced_result,
    "High Dimensionality": high_dimensionality_unbalanced_result,
}

### Plots for averaging strategies

In [ ]:

# compare_settings(
#     settings_unbalanced_results_wellspecified,
#     title = "Algorithm Performance for Unbalanced settings", selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
#                                                                                                      "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
#                                                                                                      "AIOLI_Suffix","AIOLI_lastit"]
# )

In [ ]:

# compare_settings(
#     settings_unbalanced_results_misspecified,
#     title = "Algorithm Performance for Unbalanced settings", selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
#                                                                                                      "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
#                                                                                                      "AIOLI_Suffix","AIOLI_lastit"]
# )

In [ ]:

# compare_settings(
#     settings_unbalanced_results_HD,
#     title = "Algorithm Performance for Unbalanced settings", selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
#                                                                                                      "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
#                                                                                                      "AIOLI_Suffix","AIOLI_lastit"],
#     x_labels={"High Dimensional": "High Dimensional (n = 20, d = 25)", "High Dimensionality": "High Dimensionality (n = 50, d = 25)"}
# )

### General Risk Results

In [ ]:
compare_settings(
    settings_unbalanced_results_wellspecified,
    title = "Algorithm Performance for Unbalanced Settings", selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Lasso_LR","Ridge_LR","OGD_uniform","AIOLI_uniform","SMP","Ridge_SMP", "AdaGrad_uniform"]
)

In [ ]:
compare_settings(
    settings_unbalanced_results_HD,
    title = "Algorithm Performance for Unbalanced High Dimensional settings", selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Ridge_LR","Lasso_LR","OGD_uniform","AdaGrad_uniform","AIOLI_uniform","SMP","Ridge_SMP"],
    x_labels={"High Dimensional": "High Dimensional (n = 20, d = 25)", "High Dimensionality": "High Dimensionality (n = 50, d = 25)"}
)

In [ ]:
compare_settings(
    settings_unbalanced_results_misspecified,
    title = "Algorithm Performance Across Unbalanced Settings", selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Lasso_LR","Ridge_LR","OGD_uniform","AIOLI_uniform","SMP","Ridge_SMP", "AdaGrad_uniform"]
)

## Large Betas and Large X analysis

In [ ]:
# Binomial
binomial_largebetas = SimulationConfig(
    sim_setting = "binomial_x",
    n = 100,
    large_betas = True
)

binomial_largebetas_result = run_simulation(binomial_largebetas)
print_results(binomial_largebetas_result)

In [ ]:
# Well Specified
well_specified_largebetas = SimulationConfig(
    sim_setting = "well specified",
    d = 3,
    large_betas = True
)

well_specified_largebetas_result = run_simulation(well_specified_largebetas)
print_results(well_specified_largebetas_result)

In [ ]:
settings_results_largebetas = {
    "Well specified": well_specified_largebetas_result,
    "Binomial X": binomial_largebetas_result
}

In [ ]:
compare_settings(
    settings_results_largebetas, selected_algorithms= [ "OGD_uniform","AdaGrad_uniform","AIOLI_uniform",
                                                                                                     "OGD_Suffix","OGD_Last_iterate","AdaGrad_Suffix","AdaGrad_Last_iterate",
                                                                                                     "AIOLI_Suffix","AIOLI_lastit"]
)

In [ ]:
compare_settings(
    settings_results_largebetas, selected_algorithms= [ "Classical_LR", "Classical_LR_SMOTE","Ridge_LR","OGD_uniform","AdaGrad_uniform","AIOLI_uniform","SMP","Ridge_SMP","Lasso_LR"]
)

## Changing n

In [ ]:
hazan_changing_n = simulations_changing_n([20,50,100,200,300], hazan, filename = "hazan_changing_n")


In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP", "Ridge_LR","OGD_uniform","AdaGrad_uniform"] ,raw_data= hazan_changing_n[1])

In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP", "Ridge_LR","OGD_uniform","AdaGrad_uniform", "Classical_LR"] ,raw_data= hazan_changing_n[1])

In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP", "Ridge_LR","OGD_uniform","AdaGrad_uniform"] ,raw_data= hazan_changing_n[1])

In [ ]:
hazanLR = hazan_changing_n[1]
hazanLR[hazanLR["algorithm"] == "Classical_LR"]

In [ ]:
# high_dimensional_changing_n = simulations_changing_n([10,75,100,125,150,175], high_dimensional, filename = "high_dimensional_changing_n", high_dimensional_analysis=True)

In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP","Lasso_LR", "Ridge_LR","AdaGrad_uniform","OGD_uniform", "Classical_LR"] ,raw_data= high_dimensional_changing_n[1])

In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP","Lasso_LR", "Ridge_LR"] ,raw_data= high_dimensional_changing_n[1])

In [ ]:
hd_n20 = high_dimensional_result["test_results"]
hd_n20["n"] = 20
hd_n50 = high_dimensionality_result["test_results"]
hd_n50["n"] = 50
hd_all_n =  pd.concat([high_dimensional_changing_n[1], hd_n20,hd_n50], ignore_index=True)
hd_all_n = hd_all_n.sort_values("n")

In [ ]:
hd_all_n.to_csv("hd_all_n.csv")

In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP","Lasso_LR", "Ridge_LR","Ridge_SMP"] ,raw_data= hd_all_n, title = "Algorithm Performance vs Sample Size for High Dimensional setting")

In [ ]:
plot_simulation_results(algorithms = ["OGD_uniform","AdaGrad_uniform"] ,raw_data= hd_all_n, title = "Algorithm Performance vs Sample Size for High Dimensional setting")

In [ ]:
plot_simulation_results(algorithms = ["Classical_LR","Classical_LR_SMOTE"] ,raw_data= hd_all_n, title = "Performance of the MLE vs Sample Size for High Dimensional setting")

In [ ]:
HD_LR = hd_all_n[hd_all_n["algorithm"] == "Classical_LR"]

In [ ]:
summary_HD_LR = (
    HD_LR
    .groupby("n")["loss"]
    .agg(["mean", "median", "std", "min", "max",
         lambda x: x.quantile(0.25),
         lambda x: x.quantile(0.75)])
)
summary_HD_LR.columns = ["mean", "median", "std", "min", "max", "Q25","Q75"]

In [ ]:
summary_HD_LR

In [ ]:
HD_LR_SMOTE = hd_all_n[hd_all_n["algorithm"] == "Classical_LR_SMOTE"]
summary_HD_LR_SMOTE = (
    HD_LR_SMOTE
    .groupby("n")["loss"]
    .agg(["mean", "median", "std", "min", "max",
         lambda x: x.quantile(0.25),
         lambda x: x.quantile(0.75)])
)
summary_HD_LR_SMOTE.columns = ["mean", "median", "std", "min", "max", "Q25","Q75"]
summary_HD_LR_SMOTE

In [ ]:
a,b,c,d = generate_data("high dimensional", balanced = True, n = 125,seed = 48)
print(sum(b == 1)/len(b))
print(sum(d == 1)/len(d))

In [ ]:
logreg = LogisticRegression(penalty=None, fit_intercept=False, max_iter=1000)
lr_result = logreg.fit(a, b)
lr_result.coef_

In [ ]:
coefs = classic_logistic_regression(a,b)["Batch_estimate"]

In [ ]:
log_loss(classic_logistic_regression(a,b)["Batch_estimate"], a,b)

## Function to plot data

In [ ]:
well_specified_changing_n = simulations_changing_n([10,50,100], well_specified, filename = "well_specified_changing_n")

In [ ]:
plot_simulation_results(algorithms = ["AIOLI_uniform", "SMP", "Ridge_LR"] ,raw_data= well_specified_changing_n[1],  title= "Algorithm performance vs Sample size in well specified setting")

In [ ]:

def plot_xy_dimensions(X, y, figsize=(14, 4), title="Feature vs Label "):
    """
    X: numpy array of shape (n, d)
    y: numpy array of shape (n,1)
    """

    X = np.asarray(X)
    y = np.asarray(y)

    n, d = X.shape
    fig, axes = plt.subplots(1, d, figsize=figsize)

    # Convert labels to colors
    palette = {1: "steelblue", -1: "darkred"}

    for j in range(d):
        ax = axes[j]
        sns.scatterplot(
            x=X[:, j],
            y=y,
            hue=y,
            palette=palette,
            ax=ax,
            s=40,
            alpha=0.8,
            legend=(j == d - 1)  # only show legend on last subplot
        )
        ax.set_xlabel(f"X[:, {j}]")
        ax.set_ylabel("y")
        ax.set_title(f"Dimension {j}")

    fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()


In [ ]:
# plot_xy_dimensions(a[:,3:6],b)

# Datasets

In [ ]:
# to push

In [ ]:

# import kagglehub
# import os
#
# # Set cache to current directory
# os.environ['KAGGLEHUB_CACHE'] = os.getcwd()
#
# # Download latest version  returns the local path to the dataset folder
# path = kagglehub.dataset_download("ashrafkhan94/oil-spill")
# print("Dataset downloaded to:", path)
# print("Files:", os.listdir(path)[0])  # inspect actual filename(s)
# filename = os.listdir(path)[0]

In [ ]:
# csv_file = os.path.join(path, filename)  # adjust name based on os.listdir(path) output
# df = pd.read_csv(csv_file, header = None)

In [ ]:
# df = df.drop(df.columns[0],axis=1) # Remove first column because its for analysis based on the image number and its not one of the predictive variables.

In [ ]:
# df

In [ ]:
# trialdata = generate_data("well specified")

In [ ]:
# oil_x = df[df.columns[:-1]]

In [ ]:
# oil_y = df[df.columns[-1]]
# oil_y.replace(0, -1, inplace=True)
# oil_y

In [ ]:
# test_size = round(0.5*df.shape[0])
# oil_x_train = oil_x[test_size:]
# oil_y_train = oil_y[test_size:]
# oil_x_test = oil_x[:test_size]
# oil_y_test = oil_y[:test_size]

In [ ]:
# def evaluate_on_data(data_x,data_y):
#
#     # Separate test and train
#     test_size = round(0.5*data_x.shape[0])
#     x_train = np.array(data_x[test_size:])
#     y_train = np.array(data_y[test_size:])
#     x_test = np.array(data_x[:test_size])
#     y_test = np.array(data_y[:test_size])
#
#     d = data_x.shape[1]
#
#     # Run algorithms
#     ## Get SMOTE dataset for unbalanced cases
#     sm = SMOTE(data_x = x_train, data_y = y_train, S = 100)
#     smote_x, smote_y = sm.apply_smote()
#
#     ogd = online_gd(data_y=y_train, data_x=x_train, B=default_B)
#     adagrad = online_AdaGrad(data_y=y_train, data_x=x_train, B_vector = np.repeat(5,d))
#     aioli_model = AIOLI()
#     ai = aioli_model.fit(data_y=y_train, data_x=x_train, X=default_X, B=default_B/13)
#     classic_LR = classic_logistic_regression(data_x=x_train, data_y=y_train)
#     ridge_LR = ridge_logistic_regression(data_x=x_train, data_y=y_train)
#     lasso_LR = lasso_logistic_regression(data_x=x_train, data_y=y_train)
#     smp = sample_minmax_predictor(x_train, y_train)
#     ridge_smp = ridge_sample_minmax_predictor(x_train, y_train)
#     classic_LR_smote = classic_logistic_regression(data_x=smote_x, data_y=smote_y)
#
#     # OTB estimates
#     # OGD
#     OGD_OTB_uniform = calculate_OTB(ogd["All_Betas"], "uniform")
#     OGD_OTB_Suffix  = calculate_OTB(ogd["All_Betas"], "suffix_averaging")
#     OGD_OTB_lastit  = calculate_OTB(ogd["All_Betas"], "last_iterate")
#     # AdaGrad
#     adagrad_OTB_uniform = calculate_OTB(adagrad["All_Betas"], "uniform")
#     adagrad_OTB_Suffix  = calculate_OTB(adagrad["All_Betas"], "suffix_averaging")
#     adagrad_OTB_lastit  = calculate_OTB(adagrad["All_Betas"], "last_iterate")
#
#     ############################## TEST RESULTS ##############################
#     # OGD
#     loss_OGD_uniform_test = log_loss(OGD_OTB_uniform, x_test, y_test)
#     loss_OGD_Suffix_test  = log_loss(OGD_OTB_Suffix,  x_test, y_test)
#     loss_OGD_lastit_test  = log_loss(OGD_OTB_lastit,  x_test, y_test)
#
#     # Adagrad
#     loss_adagrad_uniform_test = log_loss(adagrad_OTB_uniform, x_test, y_test)
#     loss_adagrad_Suffix_test  = log_loss(adagrad_OTB_Suffix,  x_test, y_test)
#     loss_adagrad_lastit_test  = log_loss(adagrad_OTB_lastit,  x_test, y_test)
#
#     # AIOLI
#     loss_AIOLI_uniform_test = aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="uniform")["loss"]
#     loss_AIOLI_Suffix_test = aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="suffix_averaging")["loss"]
#     loss_AIOLI_lastit_test = aioli_model.new_data_loss(new_data_x=x_test, new_data_y=y_test, OTB_method="last_iterate")["loss"]
#
#     # Logistic Regression
#     loss_LR_test = log_loss(classic_LR["Batch_estimate"], x_test, y_test)
#     loss_LR_smote_test = log_loss(classic_LR_smote["Batch_estimate"], x_test, y_test)
#     loss_LR_Ridge_test = log_loss(ridge_LR["Batch_estimate"],   x_test, y_test)
#     loss_LR_Lasso_test = log_loss(lasso_LR["Batch_estimate"],   x_test, y_test)
#
#     # SMP
#     smp_result = smp.get_new_loss(x_test, y_test)
#     ridge_smp_result = ridge_smp.get_new_loss(x_test, y_test)
#
#     algorithms_test = {
#         "Classical_LR":  {"loss": loss_LR_test},
#         "Classical_LR_SMOTE":      {"loss": loss_LR_smote_test},
#         "Ridge_LR":      {"loss": loss_LR_Ridge_test},
#         "Lasso_LR":      {"loss": loss_LR_Lasso_test},
#         "OGD_uniform": {"loss": loss_OGD_uniform_test},
#         "OGD_Suffix":  {"loss": loss_OGD_Suffix_test},
#         "OGD_Last_iterate": {"loss": loss_OGD_lastit_test},
#         "AdaGrad_uniform": {"loss": loss_adagrad_uniform_test},
#         "AdaGrad_Suffix":  {"loss": loss_adagrad_Suffix_test},
#         "AdaGrad_Last_iterate": {"loss": loss_adagrad_lastit_test},
#         "AIOLI_uniform": {"loss": loss_AIOLI_uniform_test},
#         "AIOLI_Suffix":  {"loss": loss_AIOLI_Suffix_test},
#         "AIOLI_lastit":  {"loss": loss_AIOLI_lastit_test},
#         "SMP":      {"loss": smp_result["loss"]},
#         "Ridge_SMP":      {"loss": ridge_smp_result["loss"]}
#     }
#
#     test_set_results = []
#     for name, metrics in algorithms_test.items():
#          test_set_results.append({
#             "algorithm": name,
#             **metrics
#         })
#
#     return {
#             "test_results": pd.DataFrame(test_set_results)
#         }


In [ ]:
# type(np.array(oil_x))

In [ ]:
# evaluate_on_data(oil_x,oil_y)

# Ignore from here on, it's simply code to test some functions/debugging

## AIOLI averaging outside sigmoid

In [ ]:
# class AIOLI:
#     def __init__(self):
#         self.first_sum  = []
#         self.etas_gradients = []
#         self.x0 = [] # Initializer for minimizing algorithm
#         self.reg_lambda = 0
#         self.train_size = 0
#         self.d = 0
#         self.n = 0
#
#     # Define function to minimize for AIOLI
#     @staticmethod # Because I dont use the self
#     def parameter_function_AIOLI(b, xt, reg_lambda, first_sum, etas_gradients):
#         l_hat = np.inner(b,first_sum) + b @ etas_gradients @ b #  @ to do matrix-vector multiplication
#         # Use function np.logaddexp = log(exp(x1) + exp(x2)) to ensure numerical stability, not sure how it works
#         result = l_hat + np.logaddexp(0,np.inner(b,xt)) + np.logaddexp(0,np.inner(-b,xt)) + reg_lambda*np.linalg.norm(b)**2
#         return result
#
#     def new_point_loss(self,X,Y, OTB_method):
#         convergence_new = 0
#         beta_hat = np.zeros((self.train_size,self.d))
#         first_sum  = np.zeros(self.d)
#         etas_gradients = np.zeros((self.d,self.d))
#         for t in range(self.train_size):
#             # Minimize using data up until t-1
#             result = minimize(self.parameter_function_AIOLI, self.x0, method = "l-bfgs-b", args=(X, self.reg_lambda,first_sum, etas_gradients))
#             beta_hat[t] = result.x
#             convergence_new += result.success
#             # Update minimization elements for next round
#             first_sum += self.first_sum[t]
#             etas_gradients += self.etas_gradients[t]
#
#         # Get OTB
#         big_beta = calculate_OTB(beta_hat, OTB_method)
#
#         # Suffer loss
#         loss_point = (1/self.n)*np.sum(np.logaddexp(0,-Y*np.inner(X,beta_hat)))
#
#         return loss_point
#
#     def new_data_loss(self,new_data_x,new_data_y, OTB_method: Literal["suffix_averaging", "uniform", "last_iterate"] | None = None):
#         start_time = datetime.datetime.now()
#         new_losses = [ self.new_point_loss(X,Y, OTB_method) for (X,Y) in zip(new_data_x,new_data_y) ] # Zip pairs the rows of x and y
#         end_time = datetime.datetime.now()
#         return{
#                 "runtime" : (end_time - start_time).total_seconds(),
#                 "loss" : sum(new_losses)
#             }
#
#
#     def fit(self, data_y, data_x, B, X, OTB_method = None, OTB_average_proportion = None, minimize_method = "l-bfgs-b", dynamic_B = False):
#         """AIOLI
#         This is the function to run AIOLI.
#
#         Parameters
#         ---------
#         data_y : ndarray
#             Vector with outcome(y) values. Dimensions: n x 1
#         data_x: ndarray
#             Array with independent variables. Dimensions: n x d
#         B: float
#             Radius of the L2 ball constraint for the parameter vector beta
#         X: float
#             Radius of the L2 ball constraint for the feature vector x"""
#         # Record start time
#         start_time = datetime.datetime.now()
#
#         # Get data dimensions
#         self.n = data_x.shape[0]
#         n = self.n
#         self.d = data_x.shape[1]
#         d = self.d
#         self.x0 = np.zeros(d) # Initializer for minimizing algorithm
#         self.train_size = n
#         # Initialize vectors for storage
#         betas = np.zeros((n, d))
#         ys = np.zeros(n)
#         y_hats = np.zeros(n)
#         loss = np.zeros(n)
#         etas = np.zeros(n)
#         gradients = np.zeros((n, d))
#         self.reg_lambda = 1 / (B**2)
#
#         # Keep sums for loss functions
#         self.first_sum  = np.zeros((n, d))
#         self.etas_gradients = np.zeros((n,d,d))
#         first_sum  = np.zeros(d)
#         etas_gradients = np.zeros((d,d))
#
#         # Track convergence
#         convergence = 0
#         minimizer_gradient = 0
#
#         # Run Algorithm
#         for t in range(n):
#             # Get context vector
#             xt = data_x[t]
#
#             # Update beta parameter
#             # noinspection PyTypeChecker # This is to avoid the underlining which
#             if t == 0: # Initialize to 0 for first round
#                 betas[t] = np.zeros(d)
#             else:
#                 result = minimize(self.parameter_function_AIOLI,
#                                   np.zeros(d),
#                                   method = minimize_method,
#                                   args=(xt, self.reg_lambda,first_sum, etas_gradients),
#                                   options = {"maxiter":100000}
#                                   )
#                 betas[t] = result.x
#                 convergence += result.success
#                 minimizer_gradient += np.linalg.norm(result.jac)
#
#             # # Project beta onto set
#             # betas[t] = project_onto_l2_ball(beta_tilde,B)
#             # betas[t] = self.x0
#
#             # Generate prediction
#             y_hats[t] = np.inner(xt,betas[t])
#
#             # Observe true y from data
#             ys[t] = data_y[t]
#
#              # Compute gradient
#             gradients[t] = -ys[t]*xt*sig(-ys[t]*np.inner(xt,betas[t]))
#
#             # Estimate eta
#             etas[t] = np.exp(ys[t]*y_hats[t])/(1+B*X)
#
#             # Suffer loss
#             loss[t] = np.logaddexp(0,-ys[t]*np.inner(xt,betas[t]))
#
#             # Compute elements for AIOLI loss function
#
#             fs = gradients[t]*(1 - etas[t]*np.inner(betas[t],gradients[t]))
#             eg = (etas[t] / 2) * np.outer(gradients[t], gradients[t])
#             self.first_sum[t] = fs
#             self.etas_gradients[t] = eg
#             first_sum += fs
#             etas_gradients += eg
#             # Here this is separated and a bit reiterative because else the way the vector is stored messes up with the function. I tried multiple ways and this seemed the best one.
#
#             # Dynamic Hyperparameter tuning
#             if dynamic_B:
#                 B = max(B, np.linalg.norm(betas[t]))
#
#         # for loop ends
#
#         # Get Batch estimate
#         betas_OTB = calculate_OTB(betas, OTB_method, OTB_average_proportion)
#
#         # Get Batch estimate loss
#         loss_OTB = log_loss(betas_OTB,data_x, data_y)
#
#         end_time = datetime.datetime.now()
#
#         return{
#             "runtime" : (end_time - start_time).total_seconds(),
#             "Total_Loss" : loss_OTB,
#             "Online_Loss" : sum(loss),
#             "Batch_estimate": betas_OTB,
#             "All_Betas": betas,
#             "convergence" : convergence,
#             "minimizer_gradient" : minimizer_gradient
#         }
#


## Tests

In [ ]:
#etas = np.array([1/np.sqrt(3),1/np.sqrt(3),1/np.sqrt(3)])
#xt = np.array([1,2,3])
#betas = np.array([[2,2,5],[1,1,3],[4,5,6]])
#reg_lambda = 0.3
#gradients = np.array([[2,2,5],[1,1,3],[4,5,6]])
#b = np.array([1,2,3])
#sum_loss = 5


#(sum(etas)/2)*np.inner((b-betas[2]),gradients[2])*np.inner(gradients[2],(b-betas[2]))
#np.linalg.norm(etas)


In [ ]:
#diff = b - betas
#inner_prod = np.sum(gradients * diff,axis = 1)
#np.sum(np.sum(etas)/2 * inner_prod**2)

In [ ]:
# x0 = np.array([0,0,0])

#res = minimize(parameter_function_AIOLI, x0, method = "L-BFGS-B", args=(xt, reg_lambda, sum_loss,first_sum, etas_gradients))
#parameter_function_AIOLI(x0, xt, betas, etas, reg_lambda, sum_loss,gradients)
#res.x

In [ ]:
# diff = b - betas
# gs_bs = np.sum(gradients * diff,axis = 1)
# quadratic = np.sum(np.sum(etas)/2 * gs_bs**2)
#
# l_hat = sum_loss + np.sum(gs_bs) + quadratic
#
# l_hat + np.log(1+np.exp(np.inner(b,xt))) + np.log(1+np.exp(np.inner(-b,xt))) + reg_lambda*np.linalg.norm(b)

## Generate data

In [ ]:
# n = 1000
#
# mean = [20, 5, 2]
# cov = [[1, 0, 0], [0, 3, 0], [0,0,2]]  # diagonal covariance
# d = np.shape(cov)[1]
# y = np.random.binomial(n=1, p=0.3, size=n)
#
# x1, x2, x3 = np.random.multivariate_normal(mean, cov, size = n).T
# plt.plot(x1, x2, 'x')
# plt.axis('equal')
# plt.show()
#
# plt.plot(x1, x3, 'x')
# plt.axis('equal')
# plt.show()


In [ ]:
# x = np.random.multivariate_normal(mean, cov, size = n)
# sns.countplot(x=y)
# plt.show()


In [ ]:
# np.log(1000)

## OTB conversion proof tests

In [ ]:
# weights = np.array([0.3,0.7])
# values = np.array([2,3])
#
# print(np.sum(values))
# print(np.sum(weights*values))

In [ ]:
# def modified_generate_data(setting, balanced = True, seed = 30, n = default_n, x_means = None, B = 4, a = 1, b = -1, plot_y = False, X_hazan = 1,d = 3, hyperparameter_analysis = False, betas = None, large_betas = False, large_x = False, other_xmeans = False):
#     # Allocate space for variables
#     x = []
#     y = []
#     betas = [] if betas is None else betas
#     # betas = betas
#     rng = np.random.default_rng(seed)
#     hazan_py1 = 0
#     # Define x means based on balanced or unbalanced setting
#     x_means_simulation = [1,-2,0.83] if x_means is None else x_means
#     x_means_simulation = [-2,-1.7,1.5] if hyperparameter_analysis else x_means_simulation
#     x_means_simulation = [-0.3,-1,0.5] if other_xmeans else x_means_simulation
#     # test_size = round(default_test_ratio*n)
#     test_size = default_test_size
#     full_n = n+test_size
#     av_sigma_hidden = None
#     # if large_betas or large_x and hyperparameter_analysis:
#     #     print("Large betas/X not set for hyperparameter analysis. Data generated has default betas/X.")
#     # Generate data according to setup
#     match setting:
#         case "well specified":
#             # Generate data
#             x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
#             if hyperparameter_analysis:
#                 betas = [1.4,-2.3,-0.9] if balanced else [1.7,-0.9,1.9]
#
#             elif None in betas or len(betas) == 0:
#                 betas = [0.4,-0.3,-1.2] if balanced else [-1,-1.7,2.6] # average sigmoid balanced: 0.5, unbalanced : 0.01
#
#             betas = np.array(betas)*10 if large_betas else betas
#             x = x*10 if large_x else x
#
#             y = generate_y(full_n,x,betas = betas, seed = seed, method = "well specified")
#
#         case "high dimensional" | "hidden x":
#             # Define betas and x means to fit high dimensionality
#             betas = [2,0.8,-4.2,1.4,0.5,0.2,-1,-0.3,1.2,-0.1,0.6,0.9,-2.4,2.7,-1.4,0.5,-1,4.2,1.5,-0.2,0.4,1.2,3.6,-0.8,-1.7] if balanced else [2,1.8,-3.7,1.4,0.5,0.2,-1,-0.3,2.2,0.1,0.6,1.9,-1.4,2.7,-1.4,0.5,-1,3.2,1.5,-1.5,-2.4,1.2,3.6,-0.8,-1.2]
#             x_means_simulation = [-1.6, 2.1, 0.4, 4.3, 0.9, -8.9, 6.9, 3.1,0.4, -1.1, 1.9, 0.3, 2.4, 1.0, -6.3, 0.2,3.1, -2.9, 3.1, 4.6, -0.3, 6.9, 2.4, 1.9, 3.1]
#             if hyperparameter_analysis:
#                 x_means_simulation = [2, -4.2, 1, 2.4, -1.6, -3.8, -5, -4.1,0.2, 6, 1, -3, 4.2, -2.1,5.6, -6,6.3, 4.5, -1.3, 6.2, 2.3, 1.4, 4, -2, 2] if balanced else [2, -4.2, 1, 2.4, -1.6, -3.8, -5, -4.1,0.2, 6, 1, -3, 4.2, -2.1,5.6, -6,6.3, 5.8, -1.3, 6.2, 2.3, 1.4, 4, -2, 2.6]
#
#             # Generate x again for the high dimensional setting
#             x = generate_x(full_n,25, means = x_means_simulation, seed = seed)
#             x = x*10 if large_x else x
#             betas = np.array(betas)*10 if large_betas else betas
#
#             y = generate_y(full_n,x,betas = betas, seed = seed, method = "well specified")
#             if setting == "hidden x":
#                 av_sigma_hidden = round(np.mean( [sig(xb) for xb in np.inner(x,betas)]),3)
#                 betas = betas[:3]
#                 x = x[:,:3]
#
#         case "quadratic":
#             x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
#             if hyperparameter_analysis:
#                 betas = [1.2,1.7,-1.9] if balanced else [1,-0.6,1.2]
#
#             else:
#                 betas = [-1.44,0.5,-0.8] if balanced else [0.9,-1.1,-1.2]
#             betas = np.array(betas)*10 if large_betas else betas
#             x = x*10 if large_x else x
#             y = generate_y(full_n,x,betas = betas, seed = seed, method = "quadratic")
#         case "hazan 1-dimensional":
#             # B = np.log(full_n) if B_hazan is None else B_hazan
#             B = 1.5 if hyperparameter_analysis else 1
#             # B = 300
#             # epsilon = (B**(2/3))/(n**(2/3))
#             epsilon = 0.01
#             theta = np.sqrt(epsilon)/B
#             hazan_py1 = theta/2 + X_hazan*epsilon/B
#             xy = rng.choice([[1- theta/2,a],[theta,b]], size = full_n, p = [theta/2 + X_hazan*epsilon/B, 1- (theta/2 + X_hazan*epsilon/B)])
#             x = xy[:,0]
#             y = xy[:,1]
#             x = x.reshape(full_n,1)
#
#         case "alternating":
#             # Generate data
#             x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
#             if hyperparameter_analysis:
#                 betas = [1,2.5,0.3]
#                 betas = np.array(betas)*10 if large_betas else betas
#                 x = x*10 if large_x else x
#             elif None in betas or len(betas) == 0:
#                 betas = [2.4,0.8,-1.2]
#             y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "alternating", balanced = balanced)
#         case "sinusoidal":
#             if not balanced:
#                 print("Sinusoidal will not be studied in unbalanced case")
#             # Generate data
#
#             x = generate_x(full_n,d, means = x_means_simulation, seed = seed)
#             if hyperparameter_analysis:
#                 betas = [-1.5,-1.4,1.2]
#                 betas = np.array(betas)*10 if large_betas else betas
#                 x = x*10 if large_x else x
#             elif None in betas or len(betas) == 0:
#                 betas = [2.4,1.8,-0.7] if balanced else [-1.1,0.5,-0.8] # unbalanced average 0.325 on seed 30
#             betas = np.array(betas)*10 if large_betas else betas
#             x = x*10 if large_x else x
#             y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "sinusoidal")
#
#         case "sign":
#             # Generate data
#             x = generate_x(full_n,d, means = x_means_simulation, seed = seed, sds = np.array([1,1,1]))
#             if hyperparameter_analysis:
#                 betas = [-2.3,0.2,-1.4] if balanced else [-1.3,2.8,-3.4]
#                 betas = np.array(betas)*10 if large_betas else betas
#                 x = x*10 if large_x else x
#             elif None in betas or len(betas) == 0:
#                 betas = [-0.8,-1.4,-2.6] if balanced else [1.3,-3.7,-1.2] # average
#             betas = np.array(betas)*10 if large_betas else betas
#             x = x*10 if large_x else x
#             y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "sign")
#
#         case "binomial_x":
#             if hyperparameter_analysis:
#                 x = rng.binomial(n=6,p=0.4,size = (full_n,d))
#                 betas = [-1,1.9,0.8] if balanced else [-1,2,0.8]
#                 betas = np.array(betas)*10 if large_betas else betas
#                 x = x*10 if large_x else x
#             else:
#                 x = rng.binomial(n = 5, p = 0.7, size = (full_n,d))
#                 # betas = rng.uniform(size = d,low = -0.5,high = 0.8)
#                 betas = [1.3,0.3,-1.4] if balanced else [1.5,-1,-2.9]
#             # Just to have the correct d later
#             x_means_simulation = np.zeros(d)
#             betas = np.array(betas)*10 if large_betas else betas
#             x = x*10 if large_x else x
#             y = generate_y(full_n,x,betas = betas, seed = seed, plot = plot_y, method = "well specified")
#
#     if setting == "hazan 1-dimensional":
#         print(f"{n} (training) samples were generated using the 1-dimensional setting described by Hazan et al. (2014). With an average P(Y=1) of {hazan_py1}")
#     else:
#         d = len(x_means_simulation)
#         if setting == "sign" or setting == "alternating":
#
#             print(f"{n} (training) samples were generated in the {setting} {"balanced" if balanced else "unbalanced"} setting. Using {d} variable(s), with an average P(Y=1) of {sum(y == 1)/len(y)}{" using large betas." if large_betas else "."}{"And large x" if large_x else "."}")
#         else:
#             av_sigma = round(np.mean( [sig(xb) for xb in np.inner(x,betas)]),3) if av_sigma_hidden is None else av_sigma_hidden
#             if setting == "quadratic":
#                 av_sigma = round(np.mean( [sig(np.inner(x_t**2,betas)) for x_t in x]),3)
#             av_sin = round(np.mean( [(1/2)*np.sin(5*xb) +1/2 for xb in np.inner(x,betas)]),3)
#             print(f"{n} (training) samples were generated in the {setting} {"balanced" if balanced else "unbalanced"} setting. Using {d} variable(s), with an average P(Y=1) of {av_sin if setting == "sinusoidal" else av_sigma}{" using large betas." if large_betas else "."}{"And large x" if large_x else "."}")
#
#     # Make train/test split
#
#     x_train = x[test_size:]
#     y_train = y[test_size:]
#     x_test = x[:test_size]
#     y_test = y[:test_size]
#
#     return x_train, y_train, x_test, y_test